# Đồ án YAIB – Dự báo AKI ở ICU trên Google Colab

**Bài báo:** van de Water R, Schmidt H, Elbers P, Thoral P, Arnrich B, Rockenschaub P. *Yet Another ICU Benchmark: A Flexible Multi-Center Framework for Clinical ML.* ICLR 2024.
**Code:** https://github.com/rvandewater/YAIB (MIT), cố định ở commit `c61fad0` (nhánh development, 07/10/2026)
**Dữ liệu:** eICU-CRD Demo và MIMIC-III Demo (PhysioNet, giấy phép ODbL, không cần quyền truy cập). Dữ liệu đã được đóng gói sẵn trong thư mục `demo_data/` của repo.

| Nhiệm vụ | Nội dung | Tiêu chí chấm |
|---|---|---|
| 1 | Chạy lại 6 mô hình cho bài toán AKI trên 2 bộ demo, so với Bảng 11 của bài báo | Chạy lại mô hình (30%) |
| 2 | Huấn luyện trên eICU demo, kiểm tra trên MIMIC-III demo (kiểm định ngoài) | Điều chỉnh mô hình (15%) |
| 3 | *Thử nghiệm:* sinh lại cohort với nhãn AKI **KDIGO ≥ 2** (mức độ trung bình–nặng) rồi huấn luyện lại | Điều chỉnh mô hình (15%) |

**Mỗi lần mở phiên mới** (Colab báo ngắt kết nối, hoặc bạn quay lại sau vài giờ): máy ảo đã bị xóa, phải chạy lại **Bước 1 → Bước 3** trước khi chạy bất kỳ ô nào khác. Kết quả cũ vẫn nằm trong Drive.

**Trước khi chạy:** chọn *Runtime → Change runtime type → T4 GPU*. GPU chỉ cần cho 4 mô hình học sâu (GRU, LSTM, TCN, Transformer); hồi quy logistic và LightGBM chạy bằng CPU.

## Bước 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Không có GPU: các mô hình học sâu sẽ rất chậm"
!python --version

name, memory.total [MiB]
Tesla T4, 15360 MiB
Python 3.13.15


## Bước 2. Nối Google Drive
Log và kết quả được ghi vào `MyDrive/YAIB_logs`, nên khi phiên Colab bị ngắt thì không mất kết quả.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
LOGDIR = '/content/drive/MyDrive/YAIB_logs'
os.makedirs(LOGDIR, exist_ok=True)
print('Thư mục kết quả:', LOGDIR)

Mounted at /content/drive
Thư mục kết quả: /content/drive/MyDrive/YAIB_logs


## Bước 3. Tải code và cài môi trường
YAIB dùng `uv` để cài đúng phiên bản thư viện (PyTorch 2.6, LightGBM 4.6…) vào thư mục `.venv` riêng, không đụng đến Python của Colab. Lần đầu chạy, bước này phải tải PyTorch bản CUDA nên mất vài phút.

In [ ]:
import os
# Colab đặt sẵn MPLBACKEND cho notebook; môi trường riêng của YAIB không đọc được giá trị đó nên báo lỗi.
os.environ['MPLBACKEND'] = 'Agg'
%cd /content
![ -d YAIB ] || git clone -q -b development https://github.com/rvandewater/YAIB.git
%cd /content/YAIB
!git checkout -q c61fad0f70813cdbd815c955038ea22416c46ef5
!pip install -q uv
!uv python install 3.12
!uv sync --python 3.12 2>&1 | tail -3
!uv run icu-benchmarks -h 2>&1 | head -5

/content
/content/YAIB
Python 3.12 is already installed
Resolved 210 packages in 2ms
Checked 183 packages in 2ms
usage: icu-benchmarks [-h] -d DATA_DIR -t TASK [-n NAME] [-tn TASK_NAME]
                      [-m MODEL] [-e EXPERIMENT] [-l LOG_DIR] [-s SEED]
                      [-v | --verbose | --no-verbose] [--cpu | --no-cpu]
                      [-db | --debug | --no-debug]
                      [--reproducible | --no-reproducible]


In [ ]:
# Thí nghiệm kiểm chứng: GRU trên eICU với khoảng tìm kiếm của bản thời bài báo (YAIB v0.3.0, 6/2023).
# Chỉ đổi 3 khoảng tìm kiếm của GRU (bị nới rộng ở commit 7b1ad9d, 7/2024); mọi thứ khác giữ cấu hình hiện tại
# (30 lượt tinh chỉnh × 5 fold, rồi 5×5 fold). Kết quả ghi riêng vào MyDrive/YAIB_logs_gru_v03.
import os, re, glob, json, sqlite3, subprocess, shutil, time
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/YAIB_logs_gru_v03'
LOCAL = '/content/yaib_logs_gru_v03'
KHONG_GIAN_V03 = ['optimizer/hyperparameter.lr = (1e-5, 3e-4)',          # hiện tại: (1e-6, 1e-4, "log")
                  'model/hyperparameter.hidden_dim = (32, 256, "log")',  # hiện tại: (32, 512, "log")
                  'model/hyperparameter.layer_dim = (1, 3)']             # hiện tại: (1, 10)
os.makedirs(LOCAL, exist_ok=True)
os.makedirs(DRIVE, exist_ok=True)
%cd /content/YAIB
os.environ['MPLBACKEND'] = 'Agg'
subprocess.run(['pkill', '-f', 'icu-benchmarks train'])

gpu = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip()
assert gpu, 'CHƯA CÓ GPU: chọn Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU, rồi chạy lại ô Bước 3 và ô này.'
print('GPU:', gpu)

# Sửa lỗi 2 ca eICU demo thiếu giới tính (xem Nhiệm vụ 1)
for src in ['eicu_demo', 'mimic_demo']:
    f = f'demo_data/aki/{src}/sta.parquet'
    sta = pd.read_parquet(f)
    n = int(sta['sex'].isna().sum())
    if n:
        sta['sex'] = sta['sex'].fillna(sta['sex'].mode()[0])
        sta.to_parquet(f, index=False)
        shutil.rmtree(f'demo_data/aki/{src}/cache', ignore_errors=True)
        shutil.rmtree(f'demo_data/aki/{src}/preproc', ignore_errors=True)
    print(f'{src}: đã điền {n} ca thiếu giới tính')

def dong_bo(src, dst, ghi_de=True):
    """Chép kết quả từ src sang dst, bỏ file mô hình .ckpt và thư mục tạm. Trả về số file chép lỗi."""
    loi = 0
    for root, dirs, files in os.walk(src):
        dirs[:] = [d for d in dirs if not d.startswith('tmp')]
        out = os.path.join(dst, os.path.relpath(root, src))
        os.makedirs(out, exist_ok=True)
        for fn in files:
            if fn.endswith('.ckpt'):
                continue
            s, d = os.path.join(root, fn), os.path.join(out, fn)
            try:
                if os.path.exists(d) and (not ghi_de or (not fn.endswith('.db') and os.path.getsize(d) == os.path.getsize(s)
                                                         and os.path.getmtime(d) >= os.path.getmtime(s) - 2)):
                    continue
                if fn.endswith('.db'):  # file SQLite (nhỏ, có thể đang được ghi): luôn sao lưu qua API của SQLite
                    a, b = sqlite3.connect(s), sqlite3.connect('/tmp/_ban_sao.db')
                    a.backup(b)
                    a.close(); b.close()
                    shutil.copyfile('/tmp/_ban_sao.db', d)
                else:
                    shutil.copy2(s, d)
            except Exception:
                loi += 1
    return loi

def so_fold(run):
    return len(glob.glob(os.path.join(run, 'repetition_*', 'fold_*', 'test_metrics.json'))) if run else 0

def done_folds(root, name, model):
    runs = glob.glob(os.path.join(root, name, 'AKI', model, '*', 'accumulated_test_metrics.json'))
    return max([so_fold(os.path.dirname(r)) for r in runs], default=0)

def best_db(base):
    best, best_n = None, 0
    for db in glob.glob(os.path.join(base, '*', 'hyperparameter_tuning_logs.db')):
        try:
            n = sqlite3.connect(db).execute("select count(*) from trials where state='COMPLETE'").fetchone()[0]
        except Exception:
            n = 0
        if best is None or n > best_n:
            best, best_n = db, n
    return best, best_n

def khoang(db):
    """Khoảng tốc độ học và số tầng lớn nhất đã thử, để kiểm tra khoảng tìm kiếm mới có được áp dụng."""
    try:
        con = sqlite3.connect(db)
        lo, hi = con.execute("select min(param_value), max(param_value) from trial_params where param_name like '%lr'").fetchone()
        tang = con.execute("select max(param_value) from trial_params where param_name like '%layer_dim'").fetchone()[0]
        return f'(đã thử lr {lo:.1e}–{hi:.1e}, tối đa {int(tang)} tầng)'
    except Exception:
        return ''

def gio_vn():
    return time.strftime('%H:%M', time.gmtime(time.time() + 7 * 3600))

def chay(name, model):
    sub = os.path.join(name, 'AKI', model)
    base, base_drive = os.path.join(LOCAL, sub), os.path.join(DRIVE, sub)
    dong_bo(base_drive, base, ghi_de=False)        # lấy các lượt đã có trên Drive về máy Colab
    db, n = best_db(base)
    cmd = ['uv', 'run', 'icu-benchmarks', 'train', '-d', f'demo_data/aki/{name}', '-n', name,
           '-t', 'BinaryClassification', '-tn', 'AKI', '-m', model, '--seed', '1111',
           '-l', LOCAL, '--tune', '-gc', '-lc']
    if db:
        cmd += ['--hp-checkpoint', os.path.relpath(db, base)]
    cmd += ['-hp', *KHONG_GIAN_V03]
    print(f'{gio_vn()} {name} / {model}: bắt đầu, đã có {n}/30 lượt tinh chỉnh', flush=True)
    logf = os.path.join(LOCAL, f'{name}_AKI_{model}_{time.strftime("%m%d-%H%M")}.log')
    with open(logf, 'w') as f:
        p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    p.wait(timeout=600)
                    break
                except subprocess.TimeoutExpired:
                    dong_bo(base, base_drive)          # chép sang Drive mỗi 10 phút
                    runs = sorted(glob.glob(os.path.join(base, '20*')))
                    db_now = best_db(base)
                    print(f'  {gio_vn()} tinh chỉnh {db_now[1]}/30 lượt, '
                          f'huấn luyện chính {so_fold(runs[-1]) if runs else 0}/25 fold {khoang(db_now[0])}', flush=True)
        except KeyboardInterrupt:
            subprocess.run(['pkill', '-f', 'icu-benchmarks train'])
            dong_bo(base, base_drive)
            raise
    for _ in range(5):                                 # chép lần cuối, thử lại nếu Drive báo lỗi
        if dong_bo(base, base_drive) == 0:
            break
        time.sleep(30)
    try:
        shutil.copy2(logf, DRIVE)
    except Exception:
        pass
    if p.returncode != 0:
        print(f'  LỖI (mã {p.returncode}). 30 dòng cuối của log:')
        print(''.join(open(logf).readlines()[-30:]))
        return False
    print(f'  {gio_vn()} Xong: {done_folds(DRIVE, name, model)}/25 fold đã lưu trên Drive')
    return True

for lan in range(5):
    if done_folds(DRIVE, 'eicu_demo', 'GRU') == 25:
        print('eicu_demo / GRU (khoảng tìm kiếm v0.3.0): ĐÃ XONG 25/25 fold')
        break
    if chay('eicu_demo', 'GRU'):
        break
    print(f'  {gio_vn()} chờ 1 phút rồi chạy tiếp...', flush=True)
    time.sleep(60)

# Tóm tắt, đặt cạnh bài báo và cấu hình hiện tại
runs = [os.path.dirname(p) for p in glob.glob(os.path.join(DRIVE, 'eicu_demo', 'AKI', 'GRU', '*', 'accumulated_test_metrics.json'))]
if runs:
    run = max(runs, key=lambda r: (so_fold(r), os.path.getmtime(r)))
    m = json.load(open(os.path.join(run, 'accumulated_test_metrics.json')))
    print(f"\nGRU eICU, khoảng tìm kiếm v0.3.0 ({so_fold(run)} fold):")
    print(f"  AUROC {100*m['avg']['AUC']:.1f} ± {100*m['std']['AUC']:.1f}   (bài báo 63.0 ± 0.8; cấu hình hiện tại 54.5 ± 1.4)")
    print(f"  AUPRC {100*m['avg']['PR']:.1f} ± {100*m['std']['PR']:.1f}   (bài báo 17.3 ± 0.6; cấu hình hiện tại 11.9 ± 0.6)")
    cfg = open(os.path.join(run, 'repetition_0', 'fold_0', 'train_config.gin')).read().splitlines()
    print('  Cấu hình được chọn:')
    print('\n'.join('    ' + l for l in cfg if re.search(r'hidden_dim|layer_dim|\blr\b', l)))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/YAIB
GPU: Tesla T4
eicu_demo: đã điền 0 ca thiếu giới tính
mimic_demo: đã điền 0 ca thiếu giới tính
22:35 eicu_demo / GRU: bắt đầu, đã có 0/30 lượt tinh chỉnh
  22:45 tinh chỉnh 3/30 lượt, huấn luyện chính 0/25 fold (đã thử lr 7.8e-05–1.9e-04, tối đa 3 tầng)
  22:55 tinh chỉnh 8/30 lượt, huấn luyện chính 0/25 fold (đã thử lr 3.5e-05–2.8e-04, tối đa 3 tầng)
  23:05 tinh chỉnh 11/30 lượt, huấn luyện chính 0/25 fold (đã thử lr 3.5e-05–2.8e-04, tối đa 3 tầng)
  23:15 tinh chỉnh 16/30 lượt, huấn luyện chính 0/25 fold (đã thử lr 3.5e-05–2.8e-04, tối đa 3 tầng)
  23:25 tinh chỉnh 20/30 lượt, huấn luyện chính 0/25 fold (đã thử lr 3.5e-05–2.8e-04, tối đa 3 tầng)
  23:35 tinh chỉnh 24/30 lượt, huấn luyện chính 0/25 fold (đã thử lr 3.5e-05–2.8e-04, tối đa 3 tầng)
  23:45 tinh chỉnh 26/30 lượt, huấn luyện chính 0/25 fold (đã thử lr 3.5e-05–2.8e-04, tối đa 3 tầng

## Bước 4. Xem dữ liệu demo cho bài toán AKI
Mỗi bộ có 3 file:
- `dyn.parquet`: 48 biến theo giờ, gồm creatinine (`crea`) và nước tiểu (`urine`).
- `sta.parquet`: tuổi, giới, chiều cao, cân nặng.
- `outc.parquet`: nhãn theo giờ.

**Cách YAIB gán nhãn AKI:** nhãn bằng 1 trong khoảng 6 giờ trước và 6 giờ sau lần đầu đạt KDIGO ≥ 1, và dữ liệu bị cắt ở thời điểm 6 giờ sau lần khởi phát đó.

Kết quả đúng của bước này:
- eICU demo: 1.988 lượt nằm ICU, trong đó 698 có AKI (35,1%).
- MIMIC-III demo: 113 lượt nằm ICU, trong đó 44 có AKI (38,9%).

In [ ]:
%cd /content/YAIB
import pandas as pd
for src in ['eicu_demo', 'mimic_demo']:
    o = pd.read_parquet(f'demo_data/aki/{src}/outc.parquet')
    pos = o.groupby('stay_id')['label'].max()
    print(f'{src}: {len(pos)} lượt nằm ICU, {int(pos.sum())} có AKI ({pos.mean():.1%}); '
          f'{len(o)} giờ, tỷ lệ giờ mang nhãn 1 = {o.label.mean():.1%}')

/content/YAIB
eicu_demo: 1988 lượt nằm ICU, 698 có AKI (35.1%); 78205 giờ, tỷ lệ giờ mang nhãn 1 = 11.3%
mimic_demo: 113 lượt nằm ICU, 44 có AKI (38.9%); 4968 giờ, tỷ lệ giờ mang nhãn 1 = 11.3%


## Hàm chạy YAIB và đọc kết quả

`run_yaib` gọi đúng lệnh mà tác giả dùng cho bộ demo: `--tune`, `-gc -lc`, seed 1111. Sau đó hàm đọc file `accumulated_test_metrics.json`.

- **AUC** là AUROC; **PR** là AUPRC (average precision).
- **Lưu ý khi so sánh:** con số "±" của YAIB là `pstdev / sqrt(n)`, tức sai số chuẩn chứ không phải độ lệch chuẩn.

**Chế độ nhanh cho mô hình học sâu (`QUICK_DL = True`).** Cấu hình gốc tinh chỉnh 30 lần trên 5 fold và lặp 5 × 5 cross-validation, trên Colab sẽ mất nhiều giờ. Chế độ nhanh giảm xuống 10 lần tinh chỉnh trên 1 fold và 1 lần lặp cross-validation (5 fold). Ghi rõ chế độ đã dùng trong báo cáo. Hồi quy logistic và LightGBM luôn chạy cấu hình gốc.

In [ ]:
import os, glob, json, subprocess, time
%cd /content/YAIB
os.environ['MPLBACKEND'] = 'Agg'
assert 'LOGDIR' in globals(), 'Chưa chạy Bước 2 (nối Google Drive).'

TASK_NAME = 'AKI'
DL_MODELS = {'GRU', 'LSTM', 'TCN', 'Transformer'}
QUICK_DL = True
QUICK_BINDINGS = ['tune_hyperparameters.n_calls=10',
                  'tune_hyperparameters.folds_to_tune_on=1',
                  'execute_repeated_cv.cv_repetitions_to_train=1']

# Bảng 11 của bài báo (AKI): (AUROC, ±, AUPRC, ±)
PAPER = {
    ('eicu_demo', 'LogisticRegression'): (61.3, 0.6, 16.8, 0.4), ('mimic_demo', 'LogisticRegression'): (52.9, 2.3, 16.8, 2.0),
    ('eicu_demo', 'LGBMClassifier'):     (72.6, 0.3, 23.8, 0.4), ('mimic_demo', 'LGBMClassifier'):     (60.7, 1.7, 22.4, 1.8),
    ('eicu_demo', 'GRU'):                (63.0, 0.8, 17.3, 0.6), ('mimic_demo', 'GRU'):                (50.5, 3.3, 17.6, 1.5),
    ('eicu_demo', 'LSTM'):               (61.9, 0.9, 16.2, 0.6), ('mimic_demo', 'LSTM'):               (53.5, 2.5, 15.4, 1.3),
    ('eicu_demo', 'TCN'):                (64.5, 1.0, 17.6, 0.6), ('mimic_demo', 'TCN'):                (53.7, 2.8, 19.6, 2.0),
    ('eicu_demo', 'Transformer'):        (70.1, 0.5, 21.7, 0.7), ('mimic_demo', 'Transformer'):        (53.9, 2.0, 15.3, 1.0),
}

def latest_run(name, model, task_name=TASK_NAME, sub=''):
    pattern = os.path.join(LOGDIR, name, task_name, model, sub, '*', 'accumulated_test_metrics.json')
    runs = sorted(glob.glob(pattern), key=os.path.getmtime)
    return os.path.dirname(runs[-1]) if runs else None

def read_metrics(run_dir):
    m = json.load(open(os.path.join(run_dir, 'accumulated_test_metrics.json')))
    return {'AUROC': 100 * m['avg']['AUC'], 'AUROC ±': 100 * m['std']['AUC'],
            'AUPRC': 100 * m['avg']['PR'],  'AUPRC ±': 100 * m['std']['PR'],
            'thời gian (phút)': round(m.get('execution_time', 0) / 60, 1)}

def run_yaib(data_dir, name, model, extra=(), task_name=TASK_NAME):
    cmd = ['uv', 'run', 'icu-benchmarks', 'train', '-d', data_dir, '-n', name,
           '-t', 'BinaryClassification', '-tn', task_name, '-m', model,
           '--seed', '1111', '-l', LOGDIR, '--tune', '-gc', '-lc', *extra]
    if model in DL_MODELS and QUICK_DL:
        cmd += ['-hp', *QUICK_BINDINGS]
    logf = os.path.join(LOGDIR, f'{name}_{task_name}_{model}_{time.strftime("%m%d-%H%M")}.log')
    print('Chạy:', ' '.join(cmd))
    with open(logf, 'w') as f:
        p = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    if p.returncode != 0:
        print(f'  LỖI (mã {p.returncode}). 30 dòng cuối của log {logf}:')
        print(''.join(open(logf).readlines()[-30:]))
        return None
    run = latest_run(name, model, task_name)
    print('  Xong:', run)
    return run

/content/YAIB


In [ ]:
def resume_args(name, model, task_name=TASK_NAME):
    import sqlite3
    base = os.path.join(LOGDIR, name, task_name, model)
    best, best_n = None, -1
    for d in glob.glob(os.path.join(base, '*')):
        db = os.path.join(d, 'hyperparameter_tuning_logs.db')
        if not os.path.exists(db) or os.path.exists(os.path.join(d, 'accumulated_test_metrics.json')):
            continue
        try:
            n = sqlite3.connect(db).execute("select count(*) from trials where state='COMPLETE'").fetchone()[0]
        except Exception:
            n = 0
        if n > best_n:
            best, best_n = db, n
    if best is None:
        return ()
    rel = os.path.relpath(best, base)
    print(f'  {name} / {model}: chạy tiếp từ {rel} ({best_n} lượt thử đã xong)')
    return ('--hp-checkpoint', rel)


## Nhiệm vụ 1. Chạy lại Bảng 11 (AKI)
Nên chạy 2 mô hình nhanh trước (`LogisticRegression`, `LGBMClassifier`) để kiểm tra mọi thứ hoạt động, rồi mới thêm 4 mô hình học sâu vào `MODELS`.

Bảng kết quả được lưu vào `YAIB_logs/nv1_bang11_aki.csv`. Mô hình nào đã chạy xong thì lần sau ô này đọc lại kết quả từ Drive, không chạy lại.

In [ ]:
# Bảng kết quả Nhiệm vụ 1: 6 mô hình × 2 bộ dữ liệu, so với Bảng 11 của bài báo (± là sai số chuẩn)
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, sqlite3
import pandas as pd

LOGDIR = '/content/drive/MyDrive/YAIB_logs'
PAPER = {  # Bảng 11 (AKI): AUROC, ±, AUPRC, ±
    ('eicu_demo', 'LogisticRegression'): (61.3, 0.6, 16.8, 0.4), ('eicu_demo', 'LGBMClassifier'): (72.6, 0.3, 23.8, 0.4),
    ('eicu_demo', 'GRU'): (63.0, 0.8, 17.3, 0.6),                ('eicu_demo', 'LSTM'): (61.9, 0.9, 16.2, 0.6),
    ('eicu_demo', 'TCN'): (64.5, 1.0, 17.6, 0.6),                ('eicu_demo', 'Transformer'): (70.1, 0.5, 21.7, 0.7),
    ('mimic_demo', 'LogisticRegression'): (52.9, 2.3, 16.8, 2.0), ('mimic_demo', 'LGBMClassifier'): (60.7, 1.7, 22.4, 1.8),
    ('mimic_demo', 'GRU'): (50.5, 3.3, 17.6, 1.5),                ('mimic_demo', 'LSTM'): (53.5, 2.5, 15.4, 1.3),
    ('mimic_demo', 'TCN'): (53.7, 2.8, 19.6, 2.0),                ('mimic_demo', 'Transformer'): (53.9, 2.0, 15.3, 1.0),
}

def so_fold(run):
    return len(glob.glob(os.path.join(run, 'repetition_*', 'fold_*', 'test_metrics.json')))

def so_luot(name, model):
    best = 0
    for db in glob.glob(os.path.join(LOGDIR, name, 'AKI', model, '*', 'hyperparameter_tuning_logs.db')):
        try:
            best = max(best, sqlite3.connect(db).execute("select count(*) from trials where state='COMPLETE'").fetchone()[0])
        except Exception:
            pass
    return best

rows = []
for (name, model), pa in PAPER.items():
    runs = [os.path.dirname(p) for p in glob.glob(os.path.join(LOGDIR, name, 'AKI', model, '*', 'accumulated_test_metrics.json'))]
    if not runs:
        continue
    run = max(runs, key=lambda r: (so_fold(r), os.path.getmtime(r)))
    m = json.load(open(os.path.join(run, 'accumulated_test_metrics.json')))
    auc, pr = 100 * m['avg']['AUC'], 100 * m['avg']['PR']
    rows.append({'bộ dữ liệu': name, 'mô hình': model, 'số fold': so_fold(run), 'lượt tinh chỉnh': so_luot(name, model),
                 'AUROC chạy lại': f"{auc:.1f} ± {100*m['std']['AUC']:.1f}", 'AUROC bài báo': f'{pa[0]} ± {pa[1]}',
                 'Δ AUROC': round(auc - pa[0], 1),
                 'AUPRC chạy lại': f"{pr:.1f} ± {100*m['std']['PR']:.1f}", 'AUPRC bài báo': f'{pa[2]} ± {pa[3]}',
                 'Δ AUPRC': round(pr - pa[2], 1)})
nv1 = pd.DataFrame(rows)
nv1.to_csv(os.path.join(LOGDIR, 'nv1_bang11_aki.csv'), index=False)
print(nv1.to_string(index=False))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bộ dữ liệu            mô hình  số fold  lượt tinh chỉnh AUROC chạy lại AUROC bài báo  Δ AUROC AUPRC chạy lại AUPRC bài báo  Δ AUPRC
 eicu_demo LogisticRegression       25               27     67.5 ± 0.4    61.3 ± 0.6      6.2     20.0 ± 0.4    16.8 ± 0.4      3.2
 eicu_demo     LGBMClassifier       25               30     76.9 ± 0.4    72.6 ± 0.3      4.3     31.1 ± 0.6    23.8 ± 0.4      7.3
 eicu_demo                GRU       25               30     54.5 ± 1.4    63.0 ± 0.8     -8.5     11.9 ± 0.6    17.3 ± 0.6     -5.4
 eicu_demo               LSTM       25               30     65.4 ± 0.7    61.9 ± 0.9      3.5     18.3 ± 0.5    16.2 ± 0.6      2.1
 eicu_demo                TCN       25               28     65.7 ± 0.8    64.5 ± 1.0      1.2     17.5 ± 0.5    17.6 ± 0.6     -0.1
 eicu_demo        Transformer       25               27     70.7 ± 0.5    70.1 

In [ ]:
# Kiểm tra GRU trên eICU (so với LSTM): cấu hình được chọn, các lượt tinh chỉnh tốt nhất, AUROC từng fold
from google.colab import drive
drive.mount('/content/drive')
import os, re, glob, json, sqlite3
import pandas as pd
LOGDIR = '/content/drive/MyDrive/YAIB_logs'

def so_fold(run):
    return len(glob.glob(os.path.join(run, 'repetition_*', 'fold_*', 'test_metrics.json')))

def final_run(name, model):
    runs = [os.path.dirname(p) for p in glob.glob(os.path.join(LOGDIR, name, 'AKI', model, '*', 'accumulated_test_metrics.json'))]
    return max(runs, key=lambda r: (so_fold(r), os.path.getmtime(r)))

def cac_luot(name, model):
    """Các lượt tinh chỉnh trong file Optuna có nhiều lượt nhất, xếp theo loss trên tập val (thấp = tốt)."""
    best_n, rows = -1, []
    for db in glob.glob(os.path.join(LOGDIR, name, 'AKI', model, '*', 'hyperparameter_tuning_logs.db')):
        con = sqlite3.connect(db)
        n = con.execute("select count(*) from trials where state='COMPLETE'").fetchone()[0]
        if n <= best_n:
            continue
        best_n, rows = n, []
        trials = con.execute("select t.trial_id, t.number, v.value from trials t join trial_values v "
                             "on v.trial_id = t.trial_id where t.state = 'COMPLETE'").fetchall()
        for tid, num, val in trials:
            r = {'lượt': num, 'loss (val)': round(val, 4)}
            for pname, pval, dist in con.execute('select param_name, param_value, distribution_json '
                                                 'from trial_params where trial_id = ?', (tid,)).fetchall():
                d = json.loads(dist)
                if 'Categorical' in d['name']:
                    pval = d['attributes']['choices'][int(pval)]
                elif 'Int' in d['name']:
                    pval = int(pval)
                else:
                    pval = float(f'{pval:.3g}')
                r[pname.split('/')[-1]] = pval
            rows.append(r)
    return pd.DataFrame(rows).sort_values('loss (val)')

def tung_fold(run):
    M = pd.DataFrame(index=[f'lặp {r}' for r in range(5)], columns=[f'fold {f}' for f in range(5)], dtype=float)
    for r in range(5):
        for f in range(5):
            p = os.path.join(run, f'repetition_{r}', f'fold_{f}', 'test_metrics.json')
            if os.path.exists(p):
                M.iloc[r, f] = round(100 * json.load(open(p))['AUC'], 1)
    return M

for model in ['GRU', 'LSTM']:
    run = final_run('eicu_demo', model)
    print(f'\n================ eICU / {model} ({os.path.basename(run)}) ================')
    cfg = open(os.path.join(run, 'repetition_0', 'fold_0', 'train_config.gin')).read().splitlines()
    print('Cấu hình dùng cho huấn luyện chính:')
    print('\n'.join('  ' + l for l in cfg if re.search(r'hidden_dim|layer_dim|\blr\b|epochs|patience|batch_size', l)))
    t = cac_luot('eicu_demo', model)
    print(f'\n5 lượt tinh chỉnh tốt nhất (trong {len(t)} lượt):')
    print(t.head(5).to_string(index=False))
    M = tung_fold(run)
    print('\nAUROC trên tập test từng fold (hàng = lần lặp, cột = fold):')
    print(M.to_string())
    print(f'Trung bình {M.stack().mean():.1f}, thấp nhất {M.stack().min():.1f}, cao nhất {M.stack().max():.1f}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

================ eICU / GRU (2026-10-20T20-25-50) ================
Cấu hình dùng cho huấn luyện chính:
  Adam.lr = 1.4818418387632547e-06
  DLPredictionWrapper.lr = 0.002
  GRUNet.hidden_dim = 307
  GRUNet.layer_dim = 8
  model/hyperparameter.hidden_dim = (32, 512, 'log')
  model/hyperparameter.layer_dim = (1, 10)
  optimizer/hyperparameter.lr = (1e-06, 0.0001, 'log')
  train_common.batch_size = 64
  train_common.epochs = 50
  train_common.patience = 10

5 lượt tinh chỉnh tốt nhất (trong 30 lượt):
 lượt  loss (val)  Adam.lr  GRUNet.hidden_dim  GRUNet.layer_dim
    8      0.5866 0.000001                307                 8
   23      0.6090 0.000002                 32                 1
   13      0.6096 0.000001                 32                 1
   16      0.6171 0.000001                155                10
   22      0.6250 0.000002                 32  

## Nhiệm vụ 2. Kiểm định ngoài: eICU demo → MIMIC-III demo
Lấy mô hình LightGBM đã huấn luyện trên eICU demo ở Nhiệm vụ 1 (lần lặp 0, fold 0) và đánh giá thẳng trên MIMIC-III demo, không huấn luyện lại. So kết quả này với LightGBM huấn luyện ngay trên MIMIC-III demo.

Câu hỏi cho seminar: mô hình học từ 20 bệnh viện trong eICU có dùng được ở một bệnh viện khác không?

In [ ]:
# Nhiệm vụ 2. Kiểm định ngoài: LGBM huấn luyện trên eICU demo (lần lặp 0, fold 0) → đánh giá thẳng trên MIMIC-III demo.
# Chế độ --eval của YAIB (commit c61fad0) hỏng với mô hình cổ điển: lưu mô hình thành 'last.joblib' nhưng lại đọc
# 'model.joblib', và nạp mô hình trần nên gọi set_weight() bị lỗi. Ô này làm đúng việc của --eval bằng chính hàm
# tiền xử lý và hàm lấy dữ liệu của YAIB, rồi tính AUROC/AUPRC như YAIB (roc_auc_score, average_precision_score).
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, subprocess, statistics
import pandas as pd
LOGDIR = '/content/drive/MyDrive/YAIB_logs'
%cd /content/YAIB
os.environ['MPLBACKEND'] = 'Agg'

# Sửa lỗi 2 ca eICU demo thiếu giới tính, như lúc huấn luyện (xem Nhiệm vụ 1)
sta = pd.read_parquet('demo_data/aki/eicu_demo/sta.parquet')
if sta['sex'].isna().any():
    sta['sex'] = sta['sex'].fillna(sta['sex'].mode()[0])
    sta.to_parquet('demo_data/aki/eicu_demo/sta.parquet', index=False)

def so_fold(run):
    return len(glob.glob(os.path.join(run, 'repetition_*', 'fold_*', 'test_metrics.json')))

def final_run(name, model):
    runs = [os.path.dirname(p) for p in glob.glob(os.path.join(LOGDIR, name, 'AKI', model, '*', 'accumulated_test_metrics.json'))]
    return max(runs, key=lambda r: (so_fold(r), os.path.getmtime(r)))

def fold_metrics(run):
    return {(r, f): json.load(open(os.path.join(run, f'repetition_{r}', f'fold_{f}', 'test_metrics.json')))
            for r in range(5) for f in range(5)}

src_run, int_run = final_run('eicu_demo', 'LGBMClassifier'), final_run('mimic_demo', 'LGBMClassifier')
src_model = os.path.join(src_run, 'repetition_0', 'fold_0', 'last.joblib')
print('Mô hình nguồn:', src_model)

SCRIPT = r'''
import sys, json, logging
from pathlib import Path
import gin
from joblib import load
from sklearn.metrics import roc_auc_score, average_precision_score
logging.disable(logging.WARNING)
sys.path.insert(0, '/content/YAIB')
import icu_benchmarks.run  # đăng ký mọi cấu hình gin giống hệt khi chạy YAIB thật
gin.parse_config_file('configs/tasks/BinaryClassification.gin', skip_unknown=True)  # như run.py: cấu hình bài toán trước
gin.parse_config_files_and_bindings(['configs/prediction_models/LGBMClassifier.gin',
                                     'configs/tasks/BinaryClassification.gin'], None,
                                    finalize_config=False, skip_unknown=True)
from icu_benchmarks.data.split_process_data import preprocess_data
from icu_benchmarks.data.loader import PredictionPolarsDataset
from icu_benchmarks.data.constants import DataSplit
est = load(sys.argv[1])
out = {}
jobs = [('eicu_demo', 0, 0)] + [('mimic_demo', r, f) for r in range(5) for f in range(5)]
for name, r, f in jobs:
    data = preprocess_data(Path('demo_data/aki') / name, seed=1111, cv_repetitions=5, repetition_index=r,
                           cv_folds=5, fold_index=f)
    X, y, _ = PredictionPolarsDataset(data, split=DataSplit.test, ram_cache=False).get_data_and_labels()
    if X.shape[1] != est.n_features_in_:
        raise SystemExit(f'Số đặc trưng không khớp: {X.shape[1]} so với {est.n_features_in_} lúc huấn luyện')
    p = est.predict_proba(X)[:, 1]
    out[f'{name}/{r}/{f}'] = {'AUC': roc_auc_score(y, p), 'PR': average_precision_score(y, p), 'giờ': int(len(y))}
    print(name, r, f, {k: round(v, 4) for k, v in out[f'{name}/{r}/{f}'].items()}, flush=True)
json.dump(out, open(sys.argv[2], 'w'))
'''
open('/content/YAIB/nv2_eval.py', 'w').write(SCRIPT)
logf = os.path.join(LOGDIR, 'nv2_kiem_dinh_ngoai.log')
with open(logf, 'w') as f:
    p = subprocess.run(['uv', 'run', 'python', 'nv2_eval.py', src_model, '/content/nv2_out.json'],
                       stdout=f, stderr=subprocess.STDOUT)
if p.returncode != 0:
    print(f'LỖI (mã {p.returncode}). 30 dòng cuối của log:')
    print(''.join(open(logf).readlines()[-30:]))
else:
    out = json.load(open('/content/nv2_out.json'))
    se = lambda v: statistics.pstdev(v) / len(v) ** 0.5
    # 1. Kiểm tra cách tính: chấm lại mô hình nguồn trên đúng tập test eICU (lần lặp 0, fold 0) của YAIB
    yaib_auc = fold_metrics(src_run)[(0, 0)]['AUC']
    print(f"Kiểm tra cách tính (eICU, lần lặp 0, fold 0): ô này {100*out['eicu_demo/0/0']['AUC']:.2f}, "
          f"YAIB {100*yaib_auc:.2f}  → phải gần như bằng nhau\n")
    # 2. Kiểm định ngoài so với huấn luyện ngay trên MIMIC, trên cùng 25 tập test của MIMIC
    ext = [out[f'mimic_demo/{r}/{f}'] for r in range(5) for f in range(5)]
    itn = list(fold_metrics(int_run).values())
    rows = []
    for ten, ds in [('Huấn luyện và kiểm tra trên MIMIC-III demo', itn),
                    ('Huấn luyện trên eICU demo → kiểm tra trên MIMIC-III demo', ext)]:
        a, b = [100 * d['AUC'] for d in ds], [100 * d['PR'] for d in ds]
        rows.append({'kịch bản': ten, 'AUROC': f'{statistics.mean(a):.1f} ± {se(a):.1f}',
                     'AUPRC': f'{statistics.mean(b):.1f} ± {se(b):.1f}', 'số fold': len(ds)})
    dA = [100 * (e['AUC'] - i['AUC']) for e, i in zip(ext, itn)]
    dP = [100 * (e['PR'] - i['PR']) for e, i in zip(ext, itn)]
    rows.append({'kịch bản': 'Chênh lệch theo cặp (eICU→MIMIC trừ MIMIC→MIMIC)',
                 'AUROC': f'{statistics.mean(dA):+.1f} ± {se(dA):.1f}', 'AUPRC': f'{statistics.mean(dP):+.1f} ± {se(dP):.1f}',
                 'số fold': len(dA)})
    nv2 = pd.DataFrame(rows)
    nv2.to_csv(os.path.join(LOGDIR, 'nv2_kiem_dinh_ngoai.csv'), index=False)
    print(nv2.to_string(index=False))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/YAIB
Mô hình nguồn: /content/drive/MyDrive/YAIB_logs/eicu_demo/AKI/LGBMClassifier/2026-10-19T09-13-11/repetition_0/fold_0/last.joblib
Kiểm tra cách tính (eICU, lần lặp 0, fold 0): ô này 74.00, YAIB 74.00  → phải gần như bằng nhau

                                                kịch bản       AUROC       AUPRC  số fold
              Huấn luyện và kiểm tra trên MIMIC-III demo  58.3 ± 1.6  18.9 ± 1.4       25
Huấn luyện trên eICU demo → kiểm tra trên MIMIC-III demo  69.5 ± 2.0  34.8 ± 3.1       25
        Chênh lệch theo cặp (eICU→MIMIC trừ MIMIC→MIMIC) +11.2 ± 2.8 +15.9 ± 3.1       25


## Nhiệm vụ 3 (thử nghiệm). Nhãn AKI mức độ trung bình–nặng: KDIGO ≥ 2

**Vì sao không tự gán lại nhãn trên file demo có sẵn:** với ca dương tính, dữ liệu demo đã bị cắt ở thời điểm 6 giờ sau lần đầu đạt KDIGO ≥ 1. Những ca tiến triển lên giai đoạn 2 muộn hơn vì vậy đã mất khỏi file. Phải sinh lại cohort từ dữ liệu demo gốc bằng repo **YAIB-cohorts** (viết bằng R, dùng gói `ricu`).

**Thay đổi duy nhất so với tác giả:** trong `ricu-extensions/callbacks/callback-kdigo.R`, hàm `aki()` có tham số `threshold = 1L`. Đổi tham số này thành `2L`.

**Phần này chưa được chạy thử.** Việc cài R và các gói (`ricu`, `arrow`, dữ liệu `mimic.demo` và `eicu.demo`) có thể mất khá lâu hoặc báo lỗi. Tác giả cũng lưu ý rằng ricu đã được sửa lỗi sau khi bài báo đăng, nên cohort sinh lại có thể lệch nhẹ so với bản gốc. Nếu không cài được trên Colab, nhóm có thể dùng Docker image của YAIB-cohorts trên máy cá nhân.

In [ ]:
!which Rscript || (apt-get -qq update && apt-get -qq install -y r-base)
!R --version | head -1
%cd /content
![ -d YAIB-cohorts ] || git clone -q https://github.com/rvandewater/YAIB-cohorts.git
%cd /content/YAIB-cohorts
!sed -i 's/aki <- function(..., threshold = 1L/aki <- function(..., threshold = 2L/' ricu-extensions/callbacks/callback-kdigo.R
!grep -n "aki <- function" ricu-extensions/callbacks/callback-kdigo.R
!mkdir -p /content/ricu_data && sed -i 's#/home/ruser/data#/content/ricu_data#' config.yaml && cat config.yaml

/usr/bin/Rscript
R version 4.6.1 (2026-06-24) -- "Happy Hop"
/content
/content/YAIB-cohorts
124:aki <- function(..., threshold = 2L, interval = NULL, keep_components = FALSE) {
ricu_data_path: "/content/ricu_data"
out_dir: "../data" 


In [ ]:
%%bash
# Cài gói R. Dùng kho nhị phân của Posit cho Ubuntu để khỏi phải biên dịch arrow.
CODENAME=$(lsb_release -cs)
cat > /tmp/install.R << EOF
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/${CODENAME}/latest"), Ncpus = 2)
pk <- c("ricu", "arrow", "argparser", "assertthat", "rlang", "data.table", "vctrs", "yaml", "remotes")
install.packages(setdiff(pk, rownames(installed.packages())))
if (!requireNamespace("mimic.demo", quietly = TRUE)) remotes::install_github("eth-mds/mimic-demo")
if (!requireNamespace("eicu.demo", quietly = TRUE))  remotes::install_github("eth-mds/eicu-demo")
# YAIB-cohorts dùng bản ricu vá riêng của tác giả (theo renv.lock), không phải bản CRAN
remotes::install_github("prockenschaub/ricu-package@monkeypatch0.6.0", upgrade = "never", force = TRUE)
cat("ricu", as.character(packageVersion("ricu")), "\n")
EOF
Rscript /tmp/install.R 2>&1 | tail -8

*** copying figures
** building package indices
** installing vignettes
** testing if installed package can be loaded from temporary location
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (ricu)
ricu 0.6.0 


In [ ]:
# Nhiệm vụ 3, ô 1/2. Sinh lại cohort AKI bằng YAIB-cohorts (R + ricu) với 2 ngưỡng:
#   KDIGO ≥ 1: đối chứng, cùng định nghĩa với dữ liệu demo gốc, để tách ảnh hưởng của việc sinh lại cohort
#   KDIGO ≥ 2: AKI mức độ trung bình–nặng (thay đổi duy nhất: threshold = 2L trong hàm aki() của callback-kdigo.R)
import os, subprocess, shutil
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
LOGDIR = '/content/drive/MyDrive/YAIB_logs'
COH = '/content/YAIB-cohorts'
NV3 = '/content/YAIB/demo_data_nv3'

def sh(cmd, logname, cwd=None):
    """Chạy lệnh bash, ghi log ra Drive; nếu lỗi thì in 30 dòng cuối rồi dừng."""
    logf = os.path.join(LOGDIR, logname)
    with open(logf, 'w') as f:
        p = subprocess.run(cmd, shell=True, executable='/bin/bash', cwd=cwd, stdout=f, stderr=subprocess.STDOUT)
    if p.returncode != 0:
        print(f'LỖI (mã {p.returncode}) ở bước {logname}. 30 dòng cuối của log:')
        print(''.join(open(logf).readlines()[-30:]))
        raise RuntimeError('Dừng ở bước trên. Gửi thông báo lỗi này cho người hướng dẫn.')
    return open(logf).read()

# Bước 1. R và thư viện hệ thống (units cần libudunits2)
print('1/4 Cài R...', flush=True)
out = sh('apt-get -qq update && (which Rscript || apt-get -qq install -y r-base) && '
         'apt-get -qq install -y libudunits2-dev libcurl4-openssl-dev libssl-dev && R --version | head -1', 'nv3_1_cai_R.log')
print('   ', out.strip().splitlines()[-1])

# Bước 2. Mã YAIB-cohorts, cố định ở commit 74ac699 (19/11/2025)
sh(f'[ -d {COH} ] || git clone -q https://github.com/rvandewater/YAIB-cohorts.git {COH}; '
   f'cd {COH} && git checkout -q 74ac699 && mkdir -p /content/ricu_data && '
   'sed -i \'s#^ricu_data_path:.*#ricu_data_path: "/content/ricu_data"#\' config.yaml && '
   # data.table >= 1.15 hiểu tên cột "...time" là "tra biến .time ngoài bảng" nên báo lỗi; đổi tên cột, không đổi logic
   "sed -i 's/[.][.][.]time/stop_time_/g' R/src/obs_time.R && grep -c stop_time_ R/src/obs_time.R", 'nv3_2_clone.log')

# Bước 3. Gói R: bản nhị phân của Posit (khỏi biên dịch arrow), ricu bản vá của tác giả (theo renv.lock), dữ liệu demo
print('2/4 Cài gói R (lần đầu mất 5–15 phút)...', flush=True)
open('/tmp/install.R', 'w').write(r'''
codename <- system("lsb_release -cs", intern = TRUE)
options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)), Ncpus = 2,
        HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
                                paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])))
pk <- c("arrow", "argparser", "assertthat", "rlang", "data.table", "vctrs", "yaml",
        # ricu chỉ "gợi ý" (Suggests) các gói này nên không tự cài, nhưng YAIB-cohorts cần (có trong renv.lock)
        "units", "progress", "pillar", "withr",
        # các gói ricu cần (Imports), để cài ricu từ mã nguồn
        "curl", "fst", "readr", "jsonlite", "prt", "tibble", "backports", "cli", "fansi", "openssl")
install.packages(setdiff(pk, rownames(installed.packages())))

# ricu: bản vá của tác giả (prockenschaub/ricu-package@monkeypatch0.6.0, commit 0969a52 theo renv.lock) cộng một sửa
# tương thích: data.table >= 1.15 gọi rbind(..., ignore.attr = TRUE) khi merge, nhưng rbind cho id_tbl của ricu 0.6.0
# không có tham số này nên coi TRUE là một bảng và báo lỗi "values must be length 1".
kiem_tra <- function() {
  out <- suppressWarnings(system2(file.path(R.home("bin"), "Rscript"), c("-e", shQuote(
    'cat(requireNamespace("ricu", quietly = TRUE) && "ignore.attr" %in% names(formals(ricu:::rbind_id_tbl)))')),
    stdout = TRUE, stderr = FALSE))
  identical(tail(out, 1), "TRUE")
}
if (!kiem_tra()) {
  tf <- tempfile(fileext = ".tar.gz")
  download.file("https://codeload.github.com/prockenschaub/ricu-package/tar.gz/0969a52747c71ae9f47051d640caa5df2d5169a0",
                tf, quiet = TRUE)
  ex <- tempfile(); untar(tf, exdir = ex); d <- list.files(ex, full.names = TRUE)[1]
  f <- file.path(d, "R", "tbl-base.R"); s <- readLines(f)
  cu  <- "rbind_id_tbl <- function(..., use.names = TRUE, fill = FALSE, idcol = NULL) {"
  moi <- "rbind_id_tbl <- function(..., use.names = TRUE, fill = FALSE, idcol = NULL, ignore.attr = FALSE) {"
  stopifnot(sum(s == cu) == 1)
  s[s == cu] <- moi
  writeLines(s, f)
  stopifnot(system2(file.path(R.home("bin"), "R"), c("CMD", "INSTALL", shQuote(d))) == 0)
}
for (d in c("mimic.demo", "eicu.demo"))
  if (!requireNamespace(d, quietly = TRUE)) install.packages(d, repos = "https://eth-mds.github.io/physionet-demo")
for (p in c("ricu", "units", "arrow", "data.table", "mimic.demo", "eicu.demo"))
  cat(p, as.character(packageVersion(p)), "\n")
ok <- kiem_tra()
cat("ricu 0969a52 + sửa rbind/ignore.attr:", ok, "\n")
if (!ok) stop("Chưa cài được ricu bản đã sửa")
''')
out = sh('Rscript /tmp/install.R', 'nv3_3_goi_R.log')
print('   ' + '\n    '.join(out.strip().splitlines()[-7:]))

# Bước 4. Sinh cohort: base_cohort.R một lần, rồi aki.R với từng ngưỡng (--no-init-file: bỏ qua renv của repo)
print('3/4 Sinh cohort...', flush=True)
R = f'{COH}/R'
for src in ['eicu_demo', 'mimic_demo']:
    sh(f'Rscript --no-init-file base_cohort.R --src {src}', f'nv3_4_base_{src}.log', cwd=R)
for thr in [1, 2]:
    sh(f"sed -i 's/aki <- function(..., threshold = [0-9]L/aki <- function(..., threshold = {thr}L/' "
       f"{COH}/ricu-extensions/callbacks/callback-kdigo.R && grep -q 'threshold = {thr}L' "
       f"{COH}/ricu-extensions/callbacks/callback-kdigo.R", f'nv3_5_nguong{thr}.log')
    for src in ['eicu_demo', 'mimic_demo']:
        sh(f'Rscript --no-init-file aki.R --src {src}', f'nv3_6_aki_kdigo{thr}_{src}.log', cwd=R)
        dst = f'{NV3}/aki_kdigo{thr}/{src}'
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(f'{COH}/data/aki/{src}', dst)
        sta = pd.read_parquet(f'{dst}/sta.parquet')   # sửa lỗi ca thiếu giới tính như Nhiệm vụ 1
        if sta['sex'].isna().any():
            sta['sex'] = sta['sex'].fillna(sta['sex'].mode()[0])
            sta.to_parquet(f'{dst}/sta.parquet', index=False)
        shutil.copytree(dst, f'{LOGDIR}/nv3_cohort/aki_kdigo{thr}/{src}', dirs_exist_ok=True)   # lưu bản sao trên Drive

# Tóm tắt cỡ mẫu
print('4/4 Cỡ mẫu các cohort:\n', flush=True)
rows = []
for src in ['eicu_demo', 'mimic_demo']:
    for ten, d in [('KDIGO ≥ 1, dữ liệu demo gốc', f'/content/YAIB/demo_data/aki/{src}'),
                   ('KDIGO ≥ 1, sinh lại', f'{NV3}/aki_kdigo1/{src}'),
                   ('KDIGO ≥ 2, sinh lại', f'{NV3}/aki_kdigo2/{src}')]:
        o = pd.read_parquet(f'{d}/outc.parquet')
        pos = o.groupby('stay_id')['label'].max()
        rows.append({'bộ dữ liệu': src, 'cohort': ten, 'số lượt ICU': len(pos), 'số ca dương': int(pos.sum()),
                     'tỷ lệ ca dương': f'{pos.mean():.1%}', 'số giờ': len(o), 'tỷ lệ giờ nhãn 1': f"{o['label'].mean():.1%}"})
nv3_n = pd.DataFrame(rows)
nv3_n.to_csv(os.path.join(LOGDIR, 'nv3_co_mau.csv'), index=False)
print(nv3_n.to_string(index=False))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1/4 Cài R...
    R version 4.6.1 (2026-06-24) -- "Happy Hop"
2/4 Cài gói R (lần đầu mất 5–15 phút)...
   ricu 0.6.0 
    units 1.0.1 
    arrow 25.0.1 
    data.table 1.18.6.1 
    mimic.demo 1.0.1 
    eicu.demo 1.0.1 
    ricu 0969a52 + sửa rbind/ignore.attr: TRUE
3/4 Sinh cohort...


In [ ]:
# Nhiệm vụ 3: bảng cỡ mẫu. Dữ liệu demo gốc lấy từ GitHub của YAIB (commit c61fad0), cohort sinh lại lấy từ bản sao trên Drive.
import os, urllib.request
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
LOGDIR = '/content/drive/MyDrive/YAIB_logs'
GOC = 'https://github.com/rvandewater/YAIB/raw/c61fad0f70813cdbd815c955038ea22416c46ef5/demo_data/aki/{}/outc.parquet'

rows = []
for src in ['eicu_demo', 'mimic_demo']:
    f = f'/tmp/goc_{src}_outc.parquet'
    if not os.path.exists(f):
        urllib.request.urlretrieve(GOC.format(src), f)
    for ten, path in [('KDIGO ≥ 1, dữ liệu demo gốc', f),
                      ('KDIGO ≥ 1, sinh lại', f'{LOGDIR}/nv3_cohort/aki_kdigo1/{src}/outc.parquet'),
                      ('KDIGO ≥ 2, sinh lại', f'{LOGDIR}/nv3_cohort/aki_kdigo2/{src}/outc.parquet')]:
        o = pd.read_parquet(path)
        pos = o.groupby('stay_id')['label'].max()
        rows.append({'bộ dữ liệu': src, 'cohort': ten, 'số lượt ICU': len(pos), 'số ca dương': int(pos.sum()),
                     'tỷ lệ ca dương': f'{pos.mean():.1%}', 'số giờ': len(o), 'tỷ lệ giờ nhãn 1': f"{o['label'].mean():.1%}"})
nv3_n = pd.DataFrame(rows)
nv3_n.to_csv(os.path.join(LOGDIR, 'nv3_co_mau.csv'), index=False)
print(nv3_n.to_string(index=False))


Mounted at /content/drive
bộ dữ liệu                      cohort  số lượt ICU  số ca dương tỷ lệ ca dương  số giờ tỷ lệ giờ nhãn 1
 eicu_demo KDIGO ≥ 1, dữ liệu demo gốc         1988          698          35.1%   78205            11.3%
 eicu_demo         KDIGO ≥ 1, sinh lại         1978          697          35.2%   77917            11.3%
 eicu_demo         KDIGO ≥ 2, sinh lại         1921          349          18.2%   90619             4.9%
mimic_demo KDIGO ≥ 1, dữ liệu demo gốc          113           44          38.9%    4968            11.3%
mimic_demo         KDIGO ≥ 1, sinh lại          113           44          38.9%    4968            11.3%
mimic_demo         KDIGO ≥ 2, sinh lại          127           26          20.5%    7590             4.3%


In [ ]:
# Nhiệm vụ 3, ô 2/2. Huấn luyện LGBM (cấu hình gốc của YAIB: 30 lượt tinh chỉnh × 5 fold, rồi 5×5 fold) trên
# cohort sinh lại, so sánh nhãn KDIGO ≥ 1 với KDIGO ≥ 2. Chỉ chạy LGBM: mô hình tốt nhất ở Nhiệm vụ 1 và chạy
# nhanh (vài chục phút), trong khi hồi quy logistic mất hơn một ngày cho mỗi cohort. Không cần GPU.
import os, glob, json, sqlite3, subprocess, shutil, time
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
LOGDIR = '/content/drive/MyDrive/YAIB_logs'
os.environ['MPLBACKEND'] = 'Agg'

def sh(cmd, logname, cwd=None):
    logf = os.path.join(LOGDIR, logname)
    with open(logf, 'w') as f:
        p = subprocess.run(cmd, shell=True, executable='/bin/bash', cwd=cwd, stdout=f, stderr=subprocess.STDOUT)
    if p.returncode != 0:
        print(f'LỖI (mã {p.returncode}) ở bước {logname}. 30 dòng cuối của log:')
        print(''.join(open(logf).readlines()[-30:]))
        raise RuntimeError('Dừng ở bước trên.')

# 0. Cài YAIB như ô Bước 3. Ô sinh cohort đã tạo sẵn thư mục /content/YAIB trống, nên phải dọn trước khi tải về.
if not os.path.isdir('/content/YAIB/.git'):
    shutil.rmtree('/content/YAIB', ignore_errors=True)
    print('Tải và cài YAIB (vài phút)...', flush=True)
sh('cd /content && ([ -d YAIB ] || git clone -q -b development https://github.com/rvandewater/YAIB.git) && cd YAIB && '
   'git checkout -q c61fad0f70813cdbd815c955038ea22416c46ef5 && pip install -q uv && uv python install 3.12 && '
   'uv sync --python 3.12 && uv run icu-benchmarks -h', 'nv3_7_cai_YAIB.log')
%cd /content/YAIB
# Cohort sinh lại: chép từ bản sao trên Drive
for thr in [1, 2]:
    shutil.copytree(f'{LOGDIR}/nv3_cohort/aki_kdigo{thr}', f'demo_data_nv3/aki_kdigo{thr}', dirs_exist_ok=True)
MODEL = 'LGBMClassifier'
COHORTS = [('AKI', 'demo_data/aki', 'KDIGO ≥ 1, dữ liệu demo gốc'),          # kết quả có sẵn từ Nhiệm vụ 1
           ('AKI_KDIGO1_sinhlai', 'demo_data_nv3/aki_kdigo1', 'KDIGO ≥ 1, sinh lại'),
           ('AKI_KDIGO2', 'demo_data_nv3/aki_kdigo2', 'KDIGO ≥ 2, sinh lại')]

def so_fold(run):
    return len(glob.glob(os.path.join(run, 'repetition_*', 'fold_*', 'test_metrics.json'))) if run else 0

def final_run(name, task):
    runs = [os.path.dirname(p) for p in glob.glob(os.path.join(LOGDIR, name, task, MODEL, '*', 'accumulated_test_metrics.json'))]
    return max(runs, key=lambda r: (so_fold(r), os.path.getmtime(r))) if runs else None

def best_db(base):
    best, best_n = None, 0
    for db in glob.glob(os.path.join(base, '*', 'hyperparameter_tuning_logs.db')):
        try:
            n = sqlite3.connect(db).execute("select count(*) from trials where state='COMPLETE'").fetchone()[0]
        except Exception:
            n = 0
        if best is None or n > best_n:
            best, best_n = db, n
    return best, best_n

def gio_vn():
    return time.strftime('%H:%M', time.gmtime(time.time() + 7 * 3600))

def chay(name, task, data_dir):
    base = os.path.join(LOGDIR, name, task, MODEL)
    db, n = best_db(base)
    cmd = ['uv', 'run', 'icu-benchmarks', 'train', '-d', f'{data_dir}/{name}', '-n', name,
           '-t', 'BinaryClassification', '-tn', task, '-m', MODEL, '--seed', '1111',
           '-l', LOGDIR, '--tune', '-gc', '-lc']
    if db:
        cmd += ['--hp-checkpoint', os.path.relpath(db, base)]
    print(f'{gio_vn()} {name} / {task}: bắt đầu, đã có {n}/30 lượt tinh chỉnh', flush=True)
    logf = os.path.join(LOGDIR, f'nv3_{name}_{task}_{time.strftime("%m%d-%H%M")}.log')
    with open(logf, 'w') as f:
        p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT)
        while True:
            try:
                p.wait(timeout=600)
                break
            except subprocess.TimeoutExpired:
                runs = sorted(glob.glob(os.path.join(base, '20*')))
                print(f'  {gio_vn()} tinh chỉnh {best_db(base)[1]}/30 lượt, '
                      f'huấn luyện chính {so_fold(runs[-1]) if runs else 0}/25 fold', flush=True)
    if p.returncode != 0:
        print(f'  LỖI (mã {p.returncode}). 30 dòng cuối của log {logf}:')
        print(''.join(open(logf).readlines()[-30:]))
        return False
    print(f'  {gio_vn()} Xong: {so_fold(final_run(name, task))}/25 fold')
    return True

rows = []
for name in ['eicu_demo', 'mimic_demo']:
    for task, data_dir, ten in COHORTS:
        o = pd.read_parquet(f'{data_dir}/{name}/outc.parquet')
        pos = o.groupby('stay_id')['label'].max()
        if task != 'AKI':
            if pos.sum() < 10:
                print(f'{name} / {ten}: chỉ có {int(pos.sum())} ca dương, quá ít để chia 5×5 fold, bỏ qua')
                continue
            for lan in range(3):
                if so_fold(final_run(name, task)) == 25 or chay(name, task, data_dir):
                    break
        run = final_run(name, task)
        if so_fold(run) != 25:
            continue
        m = json.load(open(os.path.join(run, 'accumulated_test_metrics.json')))
        rows.append({'bộ dữ liệu': name, 'cohort': ten, 'số lượt ICU': len(pos),
                     'ca dương': f'{int(pos.sum())} ({pos.mean():.1%})', 'giờ nhãn 1': f"{o['label'].mean():.1%}",
                     'AUROC': f"{100*m['avg']['AUC']:.1f} ± {100*m['std']['AUC']:.1f}",
                     'AUPRC': f"{100*m['avg']['PR']:.1f} ± {100*m['std']['PR']:.1f}", 'số fold': so_fold(run)})
nv3 = pd.DataFrame(rows)
nv3.to_csv(os.path.join(LOGDIR, 'nv3_kdigo2.csv'), index=False)
print('\n' + nv3.to_string(index=False))


Mounted at /content/drive
Tải và cài YAIB (vài phút)...
/content/YAIB
08:09 eicu_demo / AKI_KDIGO1_sinhlai: bắt đầu, đã có 0/30 lượt tinh chỉnh
  08:19 tinh chỉnh 17/30 lượt, huấn luyện chính 0/25 fold
  08:29 tinh chỉnh 30/30 lượt, huấn luyện chính 24/25 fold
  08:30 Xong: 25/25 fold
08:30 eicu_demo / AKI_KDIGO2: bắt đầu, đã có 0/30 lượt tinh chỉnh
  08:40 tinh chỉnh 16/30 lượt, huấn luyện chính 0/25 fold
  08:50 tinh chỉnh 30/30 lượt, huấn luyện chính 12/25 fold
  08:51 Xong: 25/25 fold
08:52 mimic_demo / AKI_KDIGO1_sinhlai: bắt đầu, đã có 0/30 lượt tinh chỉnh
  08:54 Xong: 25/25 fold
08:54 mimic_demo / AKI_KDIGO2: bắt đầu, đã có 0/30 lượt tinh chỉnh
  08:57 Xong: 25/25 fold

bộ dữ liệu                      cohort  số lượt ICU    ca dương giờ nhãn 1      AUROC      AUPRC  số fold
 eicu_demo KDIGO ≥ 1, dữ liệu demo gốc         1988 698 (35.1%)      11.3% 76.9 ± 0.4 31.1 ± 0.6       25
 eicu_demo         KDIGO ≥ 1, sinh lại         1978 697 (35.2%)      11.3% 76.6 ± 0.2 31.7 ± 0.7     

In [6]:
# BỔ SUNG, ô 2/3. Hai phân tích trên các mô hình LightGBM đã huấn luyện (seed 1111), không huấn luyện lại:
#  (a) Chấm riêng các giờ TRƯỚC khởi phát. Nhãn của YAIB đánh 1 cho 6 giờ trước VÀ 6 giờ sau khởi phát; ở đây bỏ các
#      giờ từ lúc khởi phát trở đi, để đo đúng khả năng báo trước. Khởi phát = giờ có nhãn 1 đầu tiên + 6 giờ.
#  (b) Kiểm định ngoài eICU → MIMIC-III cho KDIGO ≥ 2 (và làm lại cho KDIGO ≥ 1): mô hình eICU của từng fold (lần lặp r,
#      fold f) chấm trên tập kiểm tra MIMIC-III cùng chỉ số, nên dùng cả 25 mô hình eICU thay vì 1 như Nhiệm vụ 2.
# Ô này cũng kiểm tra nhãn có khớp đúng từng giờ với dự đoán không: YAIB lấy nhãn theo thứ tự bảng kết cục, còn đặc
# trưng thì sắp theo (lượt ICU, giờ); ở đây nhãn được ghép lại theo đúng (lượt ICU, giờ) rồi so với điểm YAIB đã lưu.
# Không cần GPU. Khoảng 30–40 phút. Bị ngắt thì chạy lại ô này: các fold đã chấm được lưu trên Drive và bỏ qua.
import os, json, subprocess, shutil, time, statistics, math
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
LOGDIR = '/content/drive/MyDrive/YAIB_logs'
OUT = f'{LOGDIR}/bs2_truoc_khoi_phat_kiem_dinh_ngoai.json'
os.environ['MPLBACKEND'] = 'Agg'
JOBS = [['KDIGO ≥ 1, demo gốc', 'AKI', 'demo_data/aki/eicu_demo', 'demo_data/aki/mimic_demo'],
        ['KDIGO ≥ 2, sinh lại', 'AKI_KDIGO2', 'demo_data_nv3/aki_kdigo2/eicu_demo', 'demo_data_nv3/aki_kdigo2/mimic_demo']]

def gio_vn():
    return time.strftime('%H:%M', time.gmtime(time.time() + 7 * 3600))

def sh(cmd, logname, cwd=None):
    logf = os.path.join(LOGDIR, logname)
    with open(logf, 'w') as f:
        p = subprocess.run(cmd, shell=True, executable='/bin/bash', cwd=cwd, stdout=f, stderr=subprocess.STDOUT)
    if p.returncode != 0:
        print(f'LỖI (mã {p.returncode}) ở bước {logname}. 30 dòng cuối của log:')
        print(''.join(open(logf).readlines()[-30:]))
        raise RuntimeError('Dừng ở bước trên. Gửi thông báo lỗi này cho người hướng dẫn.')

# Bước 1. YAIB c61fad0, cohort sinh lại, điền 2 ca thiếu giới tính (như lúc huấn luyện)
print(f'{gio_vn()} 1/3 Cài YAIB (vài phút)...', flush=True)
if not os.path.isdir('/content/YAIB/.git'):
    shutil.rmtree('/content/YAIB', ignore_errors=True)
sh('cd /content && ([ -d YAIB ] || git clone -q -b development https://github.com/rvandewater/YAIB.git) && cd YAIB && '
   'git checkout -q c61fad0f70813cdbd815c955038ea22416c46ef5 && pip install -q uv && uv python install 3.12 && '
   'uv sync --python 3.12 && uv run icu-benchmarks -h', 'bs2_cai_YAIB.log')
os.chdir('/content/YAIB')
shutil.copytree(f'{LOGDIR}/nv3_cohort/aki_kdigo2', 'demo_data_nv3/aki_kdigo2', dirs_exist_ok=True)
for job in JOBS:
    for d in job[2:]:
        sta = pd.read_parquet(f'{d}/sta.parquet')
        if sta['sex'].isna().any():
            sta['sex'] = sta['sex'].fillna(sta['sex'].mode()[0])
            sta.to_parquet(f'{d}/sta.parquet', index=False)

# Bước 2. Chấm lại bằng chính hàm tiền xử lý và hàm lấy dữ liệu của YAIB
SCRIPT = r'''
import sys, os, glob, json, logging, warnings
from pathlib import Path
import numpy as np, pandas as pd
logging.disable(logging.WARNING)
warnings.filterwarnings('ignore')
sys.path.insert(0, '/content/YAIB')
import gin
import icu_benchmarks.run  # đăng ký mọi cấu hình gin giống khi chạy YAIB thật
gin.parse_config_file('configs/tasks/BinaryClassification.gin', skip_unknown=True)
gin.parse_config_files_and_bindings(['configs/prediction_models/LGBMClassifier.gin',
                                     'configs/tasks/BinaryClassification.gin'], None,
                                    finalize_config=False, skip_unknown=True)
from joblib import load
from sklearn.metrics import roc_auc_score, average_precision_score
from icu_benchmarks.data.split_process_data import preprocess_data
from icu_benchmarks.data.loader import PredictionPolarsDataset
from icu_benchmarks.data.constants import DataSplit, DataSegment

LOGDIR = '/content/drive/MyDrive/YAIB_logs'
JOBS, OUT = json.loads(sys.argv[1]), sys.argv[2]

def so_fold(run):
    return len(glob.glob(os.path.join(run, 'repetition_*', 'fold_*', 'test_metrics.json')))

def final_run(name, task):
    runs = [os.path.dirname(p) for p in glob.glob(os.path.join(LOGDIR, name, task, 'LGBMClassifier', '*', 'accumulated_test_metrics.json'))]
    runs = [r for r in runs if so_fold(r) == 25]
    if not runs:
        raise SystemExit(f'Không tìm thấy lần chạy đủ 25 fold: {name}/{task}')
    return max(runs, key=os.path.getmtime)

def test_rows(data_dir, r, f):
    data = preprocess_data(Path(data_dir), seed=1111, cv_repetitions=5, repetition_index=r, cv_folds=5,
                           fold_index=f, load_cache=True, generate_cache=True)
    ds = PredictionPolarsDataset(data, split=DataSplit.test, ram_cache=False)
    X, y_yaib, _ = ds.get_data_and_labels()
    # Hàng của X được sắp theo (lượt ICU, giờ): ghép nhãn theo đúng cặp đó
    feat = data[DataSplit.test][DataSegment.features].select(['stay_id', 'time']).sort(['stay_id', 'time'])
    outc = data[DataSplit.test][DataSegment.outcome].select(['stay_id', 'time', 'label'])
    lab = feat.with_row_index('i').join(outc, on=['stay_id', 'time'], how='left').sort('i')
    # Chuyển sang pandas qua danh sách Python: to_pandas() cần pyarrow, mà pyarrow của YAIB không chạy với numpy 1.26
    lab = pd.DataFrame({c: lab[c].to_list() for c in ['stay_id', 'time', 'label']})
    if not (len(lab) == len(y_yaib) == X.shape[0]):
        raise SystemExit(f'Số hàng không khớp: {len(lab)}, {len(y_yaib)}, {X.shape[0]}')
    t = lab['time']
    h = (t.dt.total_seconds() / 3600 if pd.api.types.is_timedelta64_dtype(t) else t.astype(float)).to_numpy()
    y = lab['label'].astype(float).to_numpy()
    first = pd.Series(h, index=lab['stay_id'].to_numpy())[y == 1].groupby(level=0).min()
    onset = lab['stay_id'].map(first + 6).to_numpy(dtype=float)
    pre = np.isnan(onset) | (h < onset)
    return X, y_yaib, y, pre

def metr(yt, p):
    return {'AUC': float(roc_auc_score(yt, p)), 'PR': float(average_precision_score(yt, p)),
            'n': int(len(yt)), 'pos': float(np.mean(yt))}

def evaluate(model, X, y_yaib, y, pre):
    if X.shape[1] != model.n_features_in_:
        raise SystemExit(f'Số đặc trưng không khớp: {X.shape[1]} so với {model.n_features_in_} lúc huấn luyện')
    p = model.predict_proba(X)[:, 1]
    ok = ~np.isnan(y)
    return {'yaib': metr(y_yaib, p), 'all': metr(y[ok], p[ok]), 'pre': metr(y[ok & pre], p[ok & pre]),
            'lech_nhan': float(np.mean(y_yaib[ok] != y[ok])), 'thieu_nhan': int((~ok).sum())}

res = json.load(open(OUT)) if os.path.exists(OUT) else {}
for ten, task, edir, mdir in JOBS:
    erun, mrun = final_run('eicu_demo', task), final_run('mimic_demo', task)
    for r in range(5):
        for f in range(5):
            key = f'{ten}|{r}|{f}'
            if key in res:
                continue
            fold = f'repetition_{r}/fold_{f}'
            em, mm = load(f'{erun}/{fold}/last.joblib'), load(f'{mrun}/{fold}/last.joblib')
            E, M = test_rows(edir, r, f), test_rows(mdir, r, f)
            res[key] = {'eICU → eICU': evaluate(em, *E), 'MIMIC → MIMIC': evaluate(mm, *M),
                        'eICU → MIMIC': evaluate(em, *M),
                        'luu_eicu': json.load(open(f'{erun}/{fold}/test_metrics.json'))['AUC'],
                        'luu_mimic': json.load(open(f'{mrun}/{fold}/test_metrics.json'))['AUC']}
            json.dump(res, open(OUT, 'w'))  # lưu sau mỗi fold để chạy tiếp được
            print(f"{ten} lặp {r} fold {f}: eICU {100*res[key]['eICU → eICU']['all']['AUC']:.1f}, "
                  f"trước khởi phát {100*res[key]['eICU → eICU']['pre']['AUC']:.1f}", flush=True)
print('XONG', flush=True)
'''
open('/content/YAIB/bs2_eval.py', 'w').write(SCRIPT)
logf = os.path.join(LOGDIR, f'bs2_eval_{time.strftime("%m%d-%H%M")}.log')
print(f'{gio_vn()} 2/3 Chấm lại 2 × 25 fold (mỗi fold khoảng 1 phút)...', flush=True)
with open(logf, 'w') as f:
    p = subprocess.Popen(['uv', 'run', 'python', 'bs2_eval.py', json.dumps(JOBS, ensure_ascii=False), OUT],
                         cwd='/content/YAIB', stdout=f, stderr=subprocess.STDOUT)
    while True:
        try:
            p.wait(timeout=300)
            break
        except subprocess.TimeoutExpired:
            done = len(json.load(open(OUT))) if os.path.exists(OUT) else 0
            print(f'    {gio_vn()} đã chấm {done}/50 fold', flush=True)
if p.returncode != 0:
    print(f'LỖI (mã {p.returncode}). 30 dòng cuối của log {logf}:')
    print(''.join(open(logf).readlines()[-30:]))
    raise RuntimeError('Dừng ở bước trên. Gửi thông báo lỗi này cho người hướng dẫn.')

# Bước 3. Bảng kết quả (trung bình ± sai số chuẩn qua 25 fold)
res = json.load(open(OUT))
def fmt(v):
    return f'{statistics.mean(v):.1f} ± {statistics.pstdev(v) / math.sqrt(len(v)):.1f}'
rows, kiem = [], []
for ten, task, edir, mdir in JOBS:
    keys = [f'{ten}|{r}|{f}' for r in range(5) for f in range(5) if f'{ten}|{r}|{f}' in res]
    for kb in ['eICU → eICU', 'MIMIC → MIMIC', 'eICU → MIMIC']:
        v = [res[k][kb] for k in keys]
        rows.append({'nhãn': ten, 'kịch bản': kb, 'số fold': len(v),
                     'AUROC mọi giờ': fmt([100 * x['all']['AUC'] for x in v]),
                     'AUROC trước khởi phát': fmt([100 * x['pre']['AUC'] for x in v]),
                     'AUPRC mọi giờ': fmt([100 * x['all']['PR'] for x in v]),
                     'AUPRC trước khởi phát': fmt([100 * x['pre']['PR'] for x in v]),
                     'giờ dương: mọi giờ / trước khởi phát': f"{statistics.mean([100 * x['all']['pos'] for x in v]):.1f}% / "
                                                              f"{statistics.mean([100 * x['pre']['pos'] for x in v]):.1f}%"})
    for kind in ['all', 'pre']:
        d = [100 * (res[k]['eICU → MIMIC'][kind]['AUC'] - res[k]['MIMIC → MIMIC'][kind]['AUC']) for k in keys]
        rows.append({'nhãn': ten, 'kịch bản': f'Chênh lệch theo cặp ({"mọi giờ" if kind == "all" else "trước khởi phát"})',
                     'số fold': len(d), 'AUROC mọi giờ' if kind == 'all' else 'AUROC trước khởi phát': f'{fmt(d)}'})
    for k in keys:
        kiem.append(abs(res[k]['eICU → eICU']['yaib']['AUC'] - res[k]['luu_eicu']))
        kiem.append(abs(res[k]['MIMIC → MIMIC']['yaib']['AUC'] - res[k]['luu_mimic']))
    lech = statistics.mean([res[k][kb]['lech_nhan'] for k in keys for kb in ['eICU → eICU', 'MIMIC → MIMIC']])
    print(f'Kiểm tra {ten}: tỷ lệ giờ có nhãn YAIB khác nhãn ghép theo (lượt ICU, giờ) = {100 * lech:.2f}%')
print(f'Kiểm tra cách tính: chênh lệch AUROC lớn nhất so với YAIB đã lưu = {100 * max(kiem):.3f} điểm (phải gần 0)')
bang = pd.DataFrame(rows).fillna('')
bang.to_csv(os.path.join(LOGDIR, 'bs2_truoc_khoi_phat_kiem_dinh_ngoai.csv'), index=False)
print(f'\n{gio_vn()} 3/3 Kết quả LightGBM (%, trung bình ± sai số chuẩn qua 25 fold):')
print(bang.to_string(index=False))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
23:47 1/3 Cài YAIB (vài phút)...
23:47 2/3 Chấm lại 2 × 25 fold (mỗi fold khoảng 1 phút)...
Kiểm tra KDIGO ≥ 1, demo gốc: tỷ lệ giờ có nhãn YAIB khác nhãn ghép theo (lượt ICU, giờ) = 0.00%
Kiểm tra KDIGO ≥ 2, sinh lại: tỷ lệ giờ có nhãn YAIB khác nhãn ghép theo (lượt ICU, giờ) = 0.00%
Kiểm tra cách tính: chênh lệch AUROC lớn nhất so với YAIB đã lưu = 0.000 điểm (phải gần 0)

23:52 3/3 Kết quả LightGBM (%, trung bình ± sai số chuẩn qua 25 fold):
               nhãn                              kịch bản  số fold AUROC mọi giờ AUROC trước khởi phát AUPRC mọi giờ AUPRC trước khởi phát giờ dương: mọi giờ / trước khởi phát
KDIGO ≥ 1, demo gốc                           eICU → eICU       25    76.9 ± 0.4            70.3 ± 0.5    31.1 ± 0.6            13.4 ± 0.4                         11.3% / 5.7%
KDIGO ≥ 1, demo gốc                         MIMIC → MIMIC       25    

## Xử lý lỗi thường gặp

| Hiện tượng | Cách xử lý |
|---|---|
| `uv sync` báo lỗi khi tải torch | Chạy lại ô Bước 3; thường do mạng chập chờn. |
| `Invalid task or model` | Phải chạy lệnh từ thư mục `/content/YAIB` (code đọc cấu hình theo đường dẫn tương đối `configs/`). Chạy lại `%cd /content/YAIB`. |
| Mô hình học sâu chạy quá lâu | Giữ `QUICK_DL = True`, hoặc chạy từng mô hình một. Lần chạy xong được lưu trong Drive và không phải chạy lại. |
| `No GPUs available` | Đổi runtime sang T4 GPU. Hồi quy logistic và LightGBM vẫn chạy được không cần GPU. |
| Nhiệm vụ 2 báo không tìm thấy `repetition_0/fold_0` | Mở thư mục kết quả eICU demo trong Drive, xem tên thư mục thật rồi sửa biến `source_dir`. |
| Lỗi khi cài gói R ở Nhiệm vụ 3 | Phần này là thử nghiệm. Ghi lại thông báo lỗi; nhóm có thể chuyển sang Docker image của YAIB-cohorts trên máy cá nhân. |
| Phiên Colab bị ngắt | Kết quả nằm trong `MyDrive/YAIB_logs`. Chạy lại Bước 1–3 rồi chạy tiếp; các ô kết quả tự đọc lại lần chạy cũ. |

In [7]:
# BỔ SUNG, ô 3/3. KIỂM CHỨNG: LR và LightGBM trên eICU cao hơn bài báo có phải do khác phiên bản mã và thư viện không?
# Chạy lại 2 mô hình này bằng mã YAIB ngày 26/05/2023 (commit 221b00f4) với đúng thư viện ghi ở phụ lục G.2 của bài báo
# (Python 3.10, PyTorch 2.0, LightGBM 3.3.5, scikit-learn 1.2.2). Vì sao chọn bản này: Bảng 11 có từ bản arXiv đầu tiên
# (08/06/2023) và dùng sai số chuẩn (mã có từ 08/05/2023); từ 30/05/2023 mã chấm LR/LightGBM bằng nhãn 0/1 nên AUROC
# chỉ quanh 50 (lỗi này có trong bản phát hành v0.3.0). Bản 26/05/2023 là bản cuối cùng trước lỗi đó.
# Giữ nguyên như Nhiệm vụ 1: dữ liệu demo (giống hệt từng byte), seed 1111, tinh chỉnh rồi 25 fold, điền 2 ca thiếu giới tính.
# Chỉ khác: phiên bản mã và thư viện. Đã chạy thử trên máy Mac ngày 28/10/2026: LightGBM 73,6 ± 0,5; LR 49,8 ± 0,8.
# Không cần GPU. Khoảng 1–1,5 giờ. Bị ngắt giữa chừng thì chạy lại ô này: máy chạy tiếp từ các lượt tinh chỉnh đã lưu.
import os, glob, json, subprocess, time, datetime
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

LOGDIR = '/content/drive/MyDrive/YAIB_logs/kiemchung_052023'  # kết quả mã 05/2023, để riêng
NV1DIR = '/content/drive/MyDrive/YAIB_logs'                    # kết quả Nhiệm vụ 1 (mã hiện tại, c61fad0)
COMMIT = '221b00f4'
REPO, ENV = f'/content/YAIB_{COMMIT}', '/content/env2023'
PY = f'{ENV}/bin/python'
MODELS = ['LGBMClassifier', 'LogisticRegression']             # LightGBM trước vì là phép kiểm chứng chính
N_CALLS, N_INIT = 50, 10                                       # mã 2023: 50 lượt tinh chỉnh, 10 lượt đầu bốc ngẫu nhiên
os.makedirs(LOGDIR, exist_ok=True)
ENVVARS = dict(os.environ, WANDB_MODE='disabled', MPLBACKEND='Agg')

def gio_vn():
    return (datetime.datetime.utcnow() + datetime.timedelta(hours=7)).strftime('%H:%M')

def sh(cmd, logname, cwd=None):
    """Chạy lệnh bash, ghi log ra Drive; lỗi thì in 30 dòng cuối rồi dừng."""
    logf = os.path.join(LOGDIR, logname)
    with open(logf, 'w') as f:
        p = subprocess.run(cmd, shell=True, executable='/bin/bash', cwd=cwd, stdout=f, stderr=subprocess.STDOUT)
    if p.returncode != 0:
        print(f'LỖI ở bước {logname}. 30 dòng cuối của log:')
        print(''.join(open(logf).readlines()[-30:]))
        raise RuntimeError('Dừng ở bước trên. Gửi thông báo lỗi này cho người hướng dẫn.')
    return open(logf).read()

def py(code):
    """Chạy một đoạn Python trong môi trường 2023, trong thư mục mã 05/2023."""
    p = subprocess.run([PY, '-c', code], cwd=REPO, capture_output=True, text=True, env=ENVVARS)
    if p.returncode != 0:
        print(p.stderr[-3000:])
        raise RuntimeError('Dừng ở bước trên. Gửi thông báo lỗi này cho người hướng dẫn.')
    return p.stdout.strip()

# Bước 1. Python 3.10 và thư viện năm 2023 (khoảng 5 phút, mỗi phiên Colab làm một lần)
print(f'{gio_vn()} 1/4 Cài Python 3.10 và thư viện như bài báo...', flush=True)
if not os.path.exists(f'{ENV}/.xong'):
    sh(f'pip -q install uv && uv venv -q --python 3.10 {ENV} && '
       f'uv pip install -q --python {PY} --extra-index-url https://download.pytorch.org/whl/cpu '
       '--index-strategy unsafe-best-match "torch==2.0.1+cpu" numpy==1.24.3 pandas==2.0.0 pyarrow==11.0.0 '
       'scikit-learn==1.2.2 lightgbm==3.3.5 scipy==1.10.1 scikit-optimize==0.9.0 gin-config==0.5.0 '
       'pytorch-lightning==2.0.1 torchmetrics==0.11.4 pytorch-ignite==0.4.11 tensorboard==2.12.2 '
       'wandb==0.15.0 tqdm==4.64.1 matplotlib==3.7.1 setuptools==67.8.0 && '
       # recipys: thư viện tiền xử lý của tác giả, bản 05/05/2023 (bản mới nhất vào 26/05/2023)
       f'uv pip install -q --python {PY} --no-deps '
       '"recipys @ git+https://github.com/rvandewater/ReciPies.git@391129cd209b4446e9a1803d60e15bf1b3dd7fe7" && '
       # scikit-optimize 0.9.0 còn dùng np.int, đã bị bỏ từ numpy 1.24: đổi thành int (cùng nghĩa)
       f'T=$({PY} -c "import skopt, os; print(os.path.dirname(skopt.__file__))")/space/transformers.py && '
       f'sed -i "s/astype(np\\.int)/astype(int)/g" $T && touch {ENV}/.xong', 'kc_1_cai_dat.log')

# Bước 2. Mã YAIB ngày 26/05/2023; điền 2 ca thiếu giới tính giống Nhiệm vụ 1
print(f'{gio_vn()} 2/4 Tải mã YAIB ngày 26/05/2023 (commit {COMMIT})...', flush=True)
sh(f'[ -d {REPO} ] || (curl -sSL https://codeload.github.com/rvandewater/YAIB/tar.gz/{COMMIT} | tar -xz -C /content '
   f'&& mv /content/YAIB-{COMMIT}* {REPO})', 'kc_2_tai_ma.log')
print('   ', py("import sys, torch, lightgbm, sklearn, pandas; print('Python', sys.version.split()[0], '| PyTorch', "
                "torch.__version__, '| LightGBM', lightgbm.__version__, '| scikit-learn', sklearn.__version__, "
                "'| pandas', pandas.__version__)"))
print('   ', py("import pandas as pd\n"
                "f = 'demo_data/aki/eicu_demo/sta.parquet'\n"
                "sta = pd.read_parquet(f); n = int(sta['sex'].isna().sum())\n"
                "if n:\n"
                "    sta['sex'] = sta['sex'].fillna(sta['sex'].mode()[0]); sta.to_parquet(f, index=False)\n"
                "print(f'eicu_demo: điền {n} ca thiếu giới tính (0 nghĩa là đã điền từ trước)')"))

# Bước 3. Tinh chỉnh rồi chạy 25 fold cho từng mô hình
def thu_muc(model):
    return os.path.join(LOGDIR, 'eicu_demo', 'AKI', model)

def cac_lan_chay(model):
    return sorted(glob.glob(os.path.join(thu_muc(model), '20*')))

def so_fold(run):
    return len(glob.glob(os.path.join(run, 'repetition_*', 'fold_*', 'test_metrics.json')))

def so_luot(run):
    try:
        return len(json.load(open(os.path.join(run, 'hyperparameter_tuning_logs.json')))['x_iters'])
    except Exception:
        return 0

for model in MODELS:
    runs = cac_lan_chay(model)
    if any(so_fold(r) == 25 for r in runs):
        print(f'{gio_vn()} 3/4 {model}: đã xong 25 fold, bỏ qua')
        continue
    cmd = [PY, '-m', 'icu_benchmarks.run', 'train', '-d', 'demo_data/aki/eicu_demo', '-n', 'eicu_demo',
           '-t', 'BinaryClassification', '-tn', 'AKI', '-m', model, '-s', '1111', '-l', LOGDIR, '--tune', '-gc', '-lc']
    cu = max(runs, key=so_luot, default=None)
    if cu and so_luot(cu) > 0:
        con = N_CALLS - so_luot(cu)
        cmd += ['--checkpoint', os.path.basename(cu)]
        if 0 < con < N_INIT:   # mã 2023 báo lỗi nếu số lượt còn lại ít hơn số lượt bốc ngẫu nhiên ban đầu
            cmd += ['-hp', f'tune_hyperparameters.n_initial_points={con}']
        print(f'{gio_vn()} 3/4 {model}: chạy tiếp, {so_luot(cu)}/{N_CALLS} lượt tinh chỉnh đã xong', flush=True)
    else:
        print(f'{gio_vn()} 3/4 {model}: bắt đầu ({N_CALLS} lượt tinh chỉnh, rồi 25 fold)', flush=True)
    logf = os.path.join(LOGDIR, f'kc_3_{model}.log')
    with open(logf, 'a') as f:
        p = subprocess.Popen(cmd, cwd=REPO, stdout=f, stderr=subprocess.STDOUT, env=ENVVARS)
        while p.poll() is None:
            time.sleep(600)   # 10 phút báo một lần
            runs = cac_lan_chay(model)
            if runs:
                print(f'    {gio_vn()} {model}: {max(map(so_luot, runs))}/{N_CALLS} lượt tinh chỉnh, '
                      f'{so_fold(runs[-1])}/25 fold', flush=True)
    if p.returncode != 0:
        print(f'LỖI (mã {p.returncode}). 30 dòng cuối của log:')
        print(''.join(open(logf).readlines()[-30:]))
        raise RuntimeError('Dừng ở bước trên. Gửi thông báo lỗi này cho người hướng dẫn.')
    print(f'{gio_vn()}     {model}: xong', flush=True)

# Bước 4. Bảng so sánh: bài báo (Bảng 11), mã 05/2023, mã hiện tại (Nhiệm vụ 1). ± là sai số chuẩn
PAPER = {'LGBMClassifier': (72.6, 0.3, 23.8, 0.4), 'LogisticRegression': (61.3, 0.6, 16.8, 0.4)}

def ket_qua(goc, model):
    runs = [r for r in glob.glob(os.path.join(goc, 'eicu_demo', 'AKI', model, '20*')) if so_fold(r) == 25]
    if not runs:
        return None
    m = json.load(open(os.path.join(max(runs), 'accumulated_test_metrics.json')))
    return [100 * m['avg']['AUC'], 100 * m['std']['AUC'], 100 * m['avg']['PR'], 100 * m['std']['PR']]

rows = []
for model in MODELS:
    for ten, kq in [('Bài báo (Bảng 11)', list(PAPER[model])),
                    ('Mã 26/05/2023, Colab', ket_qua(LOGDIR, model)),
                    ('Mã hiện tại (c61fad0)', ket_qua(NV1DIR, model))]:
        if kq:
            rows.append({'mô hình': model, 'phiên bản': ten,
                         'AUROC': f'{kq[0]:.1f} ± {kq[1]:.1f}', 'AUPRC': f'{kq[2]:.1f} ± {kq[3]:.1f}',
                         'Δ AUROC so với bài báo': round(kq[0] - PAPER[model][0], 1)})
bang = pd.DataFrame(rows)
bang.to_csv(os.path.join(LOGDIR, 'kiemchung_052023.csv'), index=False)
print(f'\n{gio_vn()} 4/4 Kết quả (AUROC/AUPRC %, eICU demo, AKI):')
print(bang.to_string(index=False))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
23:52 1/4 Cài Python 3.10 và thư viện như bài báo...


/tmp/ipykernel_990/3913480364.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return (datetime.datetime.utcnow() + datetime.timedelta(hours=7)).strftime('%H:%M')


23:53 2/4 Tải mã YAIB ngày 26/05/2023 (commit 221b00f4)...
    Python 3.10.22 | PyTorch 2.0.1+cpu | LightGBM 3.3.5 | scikit-learn 1.2.2 | pandas 2.0.0
    eicu_demo: điền 2 ca thiếu giới tính (0 nghĩa là đã điền từ trước)
23:53 3/4 LGBMClassifier: bắt đầu (50 lượt tinh chỉnh, rồi 25 fold)
    00:03 LGBMClassifier: 24/50 lượt tinh chỉnh, 0/25 fold
    00:13 LGBMClassifier: 50/50 lượt tinh chỉnh, 3/25 fold
    00:23 LGBMClassifier: 50/50 lượt tinh chỉnh, 23/25 fold
    00:33 LGBMClassifier: 50/50 lượt tinh chỉnh, 25/25 fold
00:33     LGBMClassifier: xong
00:33 3/4 LogisticRegression: bắt đầu (50 lượt tinh chỉnh, rồi 25 fold)
    00:43 LogisticRegression: 40/50 lượt tinh chỉnh, 0/25 fold
    00:53 LogisticRegression: 50/50 lượt tinh chỉnh, 25/25 fold
00:53     LogisticRegression: xong

00:53 4/4 Kết quả (AUROC/AUPRC %, eICU demo, AKI):
           mô hình             phiên bản      AUROC      AUPRC  Δ AUROC so với bài báo
    LGBMClassifier     Bài báo (Bảng 11) 72.6 ± 0.3 23.8 ± 0.4      

In [1]:
# BỔ SUNG, ô 1/3. Chạy lại LightGBM với thêm 2 seed (2222, 3333) để biết kết quả thay đổi bao nhiêu khi đổi cách
# chia fold và tinh chỉnh. Seed 1111 đã có từ Nhiệm vụ 1 và 3. Bốn cohort: eICU và MIMIC-III, nhãn KDIGO ≥ 1 (demo gốc)
# và KDIGO ≥ 2 (sinh lại). Cấu hình gốc của YAIB (c61fad0). Không cần GPU. Khoảng 1 giờ 40 phút.
# Kết quả lưu riêng ở MyDrive/YAIB_logs_seeds để không lẫn với các lần chạy cũ.
# Bị ngắt giữa chừng thì chạy lại ô này: máy chạy tiếp từ các lượt tinh chỉnh đã lưu.
import os, glob, json, sqlite3, subprocess, shutil, time, statistics
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
LOGDIR = '/content/drive/MyDrive/YAIB_logs'          # seed 1111 (Nhiệm vụ 1 và 3)
SEEDDIR = '/content/drive/MyDrive/YAIB_logs_seeds'   # seed 2222, 3333
SEEDS = [2222, 3333]
MODEL = 'LGBMClassifier'
COHORTS = [('mimic_demo', 'AKI', 'demo_data/aki', 'KDIGO ≥ 1, demo gốc'),     # MIMIC trước vì chạy nhanh
           ('mimic_demo', 'AKI_KDIGO2', 'demo_data_nv3/aki_kdigo2', 'KDIGO ≥ 2, sinh lại'),
           ('eicu_demo', 'AKI', 'demo_data/aki', 'KDIGO ≥ 1, demo gốc'),
           ('eicu_demo', 'AKI_KDIGO2', 'demo_data_nv3/aki_kdigo2', 'KDIGO ≥ 2, sinh lại')]
os.makedirs(SEEDDIR, exist_ok=True)
os.environ['MPLBACKEND'] = 'Agg'

def gio_vn():
    return time.strftime('%H:%M', time.gmtime(time.time() + 7 * 3600))

def sh(cmd, logname, cwd=None):
    logf = os.path.join(SEEDDIR, logname)
    with open(logf, 'w') as f:
        p = subprocess.run(cmd, shell=True, executable='/bin/bash', cwd=cwd, stdout=f, stderr=subprocess.STDOUT)
    if p.returncode != 0:
        print(f'LỖI (mã {p.returncode}) ở bước {logname}. 30 dòng cuối của log:')
        print(''.join(open(logf).readlines()[-30:]))
        raise RuntimeError('Dừng ở bước trên. Gửi thông báo lỗi này cho người hướng dẫn.')

# Bước 1. YAIB c61fad0 như các ô trước; chép cohort sinh lại từ Drive; điền 2 ca thiếu giới tính như Nhiệm vụ 1
print(f'{gio_vn()} 1/3 Cài YAIB (vài phút)...', flush=True)
if not os.path.isdir('/content/YAIB/.git'):
    shutil.rmtree('/content/YAIB', ignore_errors=True)
sh('cd /content && ([ -d YAIB ] || git clone -q -b development https://github.com/rvandewater/YAIB.git) && cd YAIB && '
   'git checkout -q c61fad0f70813cdbd815c955038ea22416c46ef5 && pip install -q uv && uv python install 3.12 && '
   'uv sync --python 3.12 && uv run icu-benchmarks -h', 'bs1_cai_YAIB.log')
os.chdir('/content/YAIB')
shutil.copytree(f'{LOGDIR}/nv3_cohort/aki_kdigo2', 'demo_data_nv3/aki_kdigo2', dirs_exist_ok=True)
for d in ['demo_data/aki/eicu_demo', 'demo_data/aki/mimic_demo',
          'demo_data_nv3/aki_kdigo2/eicu_demo', 'demo_data_nv3/aki_kdigo2/mimic_demo']:
    sta = pd.read_parquet(f'{d}/sta.parquet')
    n = int(sta['sex'].isna().sum())
    if n:
        sta['sex'] = sta['sex'].fillna(sta['sex'].mode()[0])
        sta.to_parquet(f'{d}/sta.parquet', index=False)
        shutil.rmtree(f'{d}/cache', ignore_errors=True)
        shutil.rmtree(f'{d}/preproc', ignore_errors=True)
    print(f'    {d}: điền {n} ca thiếu giới tính')

# Bước 2. Huấn luyện từng seed
def so_fold(run):
    return len(glob.glob(os.path.join(run, 'repetition_*', 'fold_*', 'test_metrics.json'))) if run else 0

def final_run(root, name, task):
    runs = [os.path.dirname(p) for p in glob.glob(os.path.join(root, name, task, MODEL, '*', 'accumulated_test_metrics.json'))]
    runs = [r for r in runs if so_fold(r) == 25]
    return max(runs, key=os.path.getmtime) if runs else None

def best_db(base):
    best, best_n = None, 0
    for db in glob.glob(os.path.join(base, '*', 'hyperparameter_tuning_logs.db')):
        try:
            n = sqlite3.connect(db).execute("select count(*) from trials where state='COMPLETE'").fetchone()[0]
        except Exception:
            n = 0
        if best is None or n > best_n:
            best, best_n = db, n
    return best, best_n

def chay(name, task, data_dir, seed):
    root = f'{SEEDDIR}/seed{seed}'
    base = os.path.join(root, name, task, MODEL)
    db, n = best_db(base)
    cmd = ['uv', 'run', 'icu-benchmarks', 'train', '-d', f'{data_dir}/{name}', '-n', name,
           '-t', 'BinaryClassification', '-tn', task, '-m', MODEL, '--seed', str(seed),
           '-l', root, '--tune', '-gc', '-lc']
    if db:
        cmd += ['--hp-checkpoint', os.path.relpath(db, base)]
    print(f'{gio_vn()} 2/3 {name} / {task} / seed {seed}: bắt đầu, đã có {n}/30 lượt tinh chỉnh', flush=True)
    logf = os.path.join(SEEDDIR, f'bs1_{name}_{task}_seed{seed}_{time.strftime("%m%d-%H%M")}.log')
    with open(logf, 'w') as f:
        p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT)
        while True:
            try:
                p.wait(timeout=600)
                break
            except subprocess.TimeoutExpired:
                runs = sorted(glob.glob(os.path.join(base, '20*')))
                print(f'    {gio_vn()} tinh chỉnh {best_db(base)[1]}/30 lượt, '
                      f'huấn luyện chính {so_fold(runs[-1]) if runs else 0}/25 fold', flush=True)
    if p.returncode != 0:
        print(f'    LỖI (mã {p.returncode}). 30 dòng cuối của log {logf}:')
        print(''.join(open(logf).readlines()[-30:]))
        return False
    print(f'    {gio_vn()} Xong: {so_fold(final_run(root, name, task))}/25 fold', flush=True)
    return True

for name, task, data_dir, ten in COHORTS:
    for seed in SEEDS:
        for lan in range(3):   # thử lại tối đa 3 lần nếu Drive lỗi tạm thời
            if final_run(f'{SEEDDIR}/seed{seed}', name, task) or chay(name, task, data_dir, seed):
                break

# Bước 3. Bảng so sánh 3 seed (± trong từng seed là sai số chuẩn qua 25 fold; cột cuối là trung bình và độ lệch chuẩn giữa 3 seed)
rows = []
for name, task, data_dir, ten in COHORTS:
    row = {'bộ dữ liệu': name, 'cohort': ten}
    auc, pr = [], []
    for seed, root in [(1111, LOGDIR)] + [(s, f'{SEEDDIR}/seed{s}') for s in SEEDS]:
        run = final_run(root, name, task)
        if not run:
            row[f'seed {seed}'] = 'chưa có'
            continue
        m = json.load(open(os.path.join(run, 'accumulated_test_metrics.json')))
        auc.append(100 * m['avg']['AUC']); pr.append(100 * m['avg']['PR'])
        row[f'seed {seed}'] = f"{auc[-1]:.1f} ± {100*m['std']['AUC']:.1f} / {pr[-1]:.1f} ± {100*m['std']['PR']:.1f}"
    if len(auc) == 3:
        row['trung bình 3 seed (SD)'] = (f'{statistics.mean(auc):.1f} ({statistics.stdev(auc):.1f}) / '
                                         f'{statistics.mean(pr):.1f} ({statistics.stdev(pr):.1f})')
    rows.append(row)
bang = pd.DataFrame(rows)
bang.to_csv(os.path.join(SEEDDIR, 'ba_seed_lgbm.csv'), index=False)
print(f'\n{gio_vn()} 3/3 LightGBM, AUROC / AUPRC (%) theo seed:')
print(bang.to_string(index=False))


Mounted at /content/drive
07:02 1/3 Cài YAIB (vài phút)...
    demo_data/aki/eicu_demo: điền 2 ca thiếu giới tính
    demo_data/aki/mimic_demo: điền 0 ca thiếu giới tính
    demo_data_nv3/aki_kdigo2/eicu_demo: điền 0 ca thiếu giới tính
    demo_data_nv3/aki_kdigo2/mimic_demo: điền 0 ca thiếu giới tính

07:05 3/3 LightGBM, AUROC / AUPRC (%) theo seed:
bộ dữ liệu              cohort               seed 1111               seed 2222               seed 3333  trung bình 3 seed (SD)
mimic_demo KDIGO ≥ 1, demo gốc 58.3 ± 1.6 / 18.9 ± 1.4 56.1 ± 2.1 / 17.8 ± 1.3 56.3 ± 2.0 / 17.3 ± 1.4 56.9 (1.2) / 18.0 (0.8)
mimic_demo KDIGO ≥ 2, sinh lại 76.5 ± 1.9 / 34.2 ± 2.0 76.7 ± 1.9 / 27.9 ± 2.2 75.1 ± 2.6 / 29.6 ± 2.5 76.1 (0.9) / 30.5 (3.3)
 eicu_demo KDIGO ≥ 1, demo gốc 76.9 ± 0.4 / 31.1 ± 0.6 77.1 ± 0.2 / 31.3 ± 0.5 77.0 ± 0.3 / 31.3 ± 0.7 77.0 (0.1) / 31.3 (0.1)
 eicu_demo KDIGO ≥ 2, sinh lại 76.6 ± 0.3 / 18.7 ± 0.4 77.2 ± 0.3 / 18.5 ± 0.6 76.3 ± 0.5 / 18.4 ± 0.9 76.7 (0.5) / 18.5 (0.1)


In [2]:
# BỔ SUNG, ô 4. KIỂM TRA: các lỗi YAIB sửa sau lượt chạy của nhóm (tháng 10/2026) có làm đổi kết quả LightGBM trên eICU không?
# Lượt A: commit 26b682d7 (23/10/2026) = c61fad0 + bản sửa biến đếm (issue #200) + một thay đổi chỉ dành cho bài hồi quy.
# Lượt B: commit 6c9c19f4 (29/10/2026), bản mới nhất: thêm các bản sửa sắp xếp dữ liệu theo lượt nằm và giờ, xử lý NaN,
#         giới tính thiếu (#251) và hai lỗi chỉ của mô hình sâu (#243, #244); lỗi thang log khi tinh chỉnh (#240) chưa sửa.
# Lưu ý: mã 05/2023 cũng tính biến đếm sau khi điền giá trị thiếu, giống c61fad0, nên #200 không giải thích được chênh lệch với
# bài báo. Ô này trả lời câu hỏi khác: số liệu LightGBM của nhóm (76,9) có bị các lỗi sửa sau này làm lệch không.
# Giữ nguyên như Nhiệm vụ 1: dữ liệu demo, seed 1111, tinh chỉnh 30 lượt rồi 25 fold, điền 2 ca thiếu giới tính.
# Không cần GPU. Khoảng 1–1,5 giờ. Bị ngắt giữa chừng thì chạy lại ô này: máy chạy tiếp từ các lượt tinh chỉnh đã lưu.
import os, glob, json, sqlite3, subprocess, shutil, time
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
LOGDIR = '/content/drive/MyDrive/YAIB_logs'                    # Nhiệm vụ 1 (c61fad0), seed 1111
KC2023 = '/content/drive/MyDrive/YAIB_logs/kiemchung_052023'   # ô 3: mã 26/05/2023
FIXDIR = '/content/drive/MyDrive/YAIB_logs_sualoi'             # ô này
COMMITS = [('26b682d7e9aff507f7f5e0e2dcde85b5a706fc99', 'A: c61fad0 + sửa #200 (23/10/2026)'),
           ('6c9c19f4d8e8f98addf062ea0171e97df82e745e', 'B: bản mới nhất (29/10/2026)')]
MODEL, NAME, TASK, SEED = 'LGBMClassifier', 'eicu_demo', 'AKI', 1111
os.makedirs(FIXDIR, exist_ok=True)
os.environ['MPLBACKEND'] = 'Agg'

def gio_vn():
    return time.strftime('%H:%M', time.gmtime(time.time() + 7 * 3600))

def sh(cmd, logname, cwd=None):
    logf = os.path.join(FIXDIR, logname)
    with open(logf, 'w') as f:
        p = subprocess.run(cmd, shell=True, executable='/bin/bash', cwd=cwd, stdout=f, stderr=subprocess.STDOUT)
    if p.returncode != 0:
        print(f'LỖI (mã {p.returncode}) ở bước {logname}. 30 dòng cuối của log:')
        print(''.join(open(logf).readlines()[-30:]))
        raise RuntimeError('Dừng ở bước trên. Gửi thông báo lỗi này cho người hướng dẫn.')

# Bước 1. Mỗi commit một thư mục riêng (để bộ nhớ đệm tiền xử lý không lẫn nhau); điền 2 ca thiếu giới tính như Nhiệm vụ 1
for sha, ten in COMMITS:
    repo = f'/content/YAIB_{sha[:8]}'
    print(f'{gio_vn()} 1/3 Cài YAIB {sha[:8]} ({ten}), vài phút...', flush=True)
    if not os.path.isdir(f'{repo}/.git'):
        shutil.rmtree(repo, ignore_errors=True)
    sh(f'cd /content && ([ -d {repo} ] || git clone -q -b development https://github.com/rvandewater/YAIB.git {repo}) && '
       f'cd {repo} && git checkout -q {sha} && pip install -q uv && uv python install 3.12 && uv sync --python 3.12 && '
       'uv run icu-benchmarks -h', f'sl1_cai_{sha[:8]}.log')
    d = f'{repo}/demo_data/aki/eicu_demo'
    sta = pd.read_parquet(f'{d}/sta.parquet')
    n = int(sta['sex'].isna().sum())
    if n:
        sta['sex'] = sta['sex'].fillna(sta['sex'].mode()[0])
        sta.to_parquet(f'{d}/sta.parquet', index=False)
        shutil.rmtree(f'{d}/cache', ignore_errors=True)
        shutil.rmtree(f'{d}/preproc', ignore_errors=True)
    print(f'    điền {n} ca thiếu giới tính (0 nghĩa là đã điền từ trước)')

# Bước 2. Tinh chỉnh rồi chạy 25 fold cho từng commit
def so_fold(run):
    return len(glob.glob(os.path.join(run, 'repetition_*', 'fold_*', 'test_metrics.json'))) if run else 0

def final_run(root):
    runs = [os.path.dirname(p) for p in glob.glob(os.path.join(root, NAME, TASK, MODEL, '*', 'accumulated_test_metrics.json'))]
    runs = [r for r in runs if so_fold(r) == 25]
    return max(runs, key=os.path.getmtime) if runs else None

def best_db(base):
    best, best_n = None, 0
    for db in glob.glob(os.path.join(base, '*', 'hyperparameter_tuning_logs.db')):
        try:
            n = sqlite3.connect(db).execute("select count(*) from trials where state='COMPLETE'").fetchone()[0]
        except Exception:
            n = 0
        if best is None or n > best_n:
            best, best_n = db, n
    return best, best_n

def chay(sha):
    repo, root = f'/content/YAIB_{sha[:8]}', f'{FIXDIR}/{sha[:8]}'
    base = os.path.join(root, NAME, TASK, MODEL)
    db, n = best_db(base)
    cmd = ['uv', 'run', 'icu-benchmarks', 'train', '-d', f'demo_data/aki/{NAME}', '-n', NAME,
           '-t', 'BinaryClassification', '-tn', TASK, '-m', MODEL, '--seed', str(SEED),
           '-l', root, '--tune', '-gc', '-lc']
    if db:
        cmd += ['--hp-checkpoint', os.path.relpath(db, base)]
    print(f'{gio_vn()} 2/3 {sha[:8]}: bắt đầu, đã có {n}/30 lượt tinh chỉnh', flush=True)
    logf = os.path.join(FIXDIR, f'sl2_{sha[:8]}_{time.strftime("%m%d-%H%M")}.log')
    with open(logf, 'w') as f:
        p = subprocess.Popen(cmd, cwd=repo, stdout=f, stderr=subprocess.STDOUT)
        while True:
            try:
                p.wait(timeout=600)
                break
            except subprocess.TimeoutExpired:
                runs = sorted(glob.glob(os.path.join(base, '20*')))
                print(f'    {gio_vn()} tinh chỉnh {best_db(base)[1]}/30 lượt, '
                      f'huấn luyện chính {so_fold(runs[-1]) if runs else 0}/25 fold', flush=True)
    if p.returncode != 0:
        print(f'    LỖI (mã {p.returncode}). 30 dòng cuối của log {logf}:')
        print(''.join(open(logf).readlines()[-30:]))
        return False
    print(f'    {gio_vn()} Xong: {so_fold(final_run(root))}/25 fold', flush=True)
    return True

for sha, ten in COMMITS:
    for lan in range(3):   # thử lại tối đa 3 lần nếu Drive lỗi tạm thời
        if final_run(f'{FIXDIR}/{sha[:8]}') or chay(sha):
            break

# Bước 3. Bảng so sánh (± là sai số chuẩn qua 25 fold) và siêu tham số được chọn
def ket_qua(root):
    run = final_run(root)
    if not run:
        return None
    m = json.load(open(os.path.join(run, 'accumulated_test_metrics.json')))
    return [100 * m['avg']['AUC'], 100 * m['std']['AUC'], 100 * m['avg']['PR'], 100 * m['std']['PR']]

def sieu_tham_so(root):
    db, n = best_db(os.path.join(root, NAME, TASK, MODEL))
    if not db:
        return '(không có cơ sở dữ liệu Optuna)'
    con = sqlite3.connect(db)
    tid = con.execute("select t.trial_id from trials t join trial_values v on t.trial_id = v.trial_id "
                      "where t.state='COMPLETE' order by v.value limit 1").fetchone()[0]
    ps = con.execute('select param_name, param_value from trial_params where trial_id = ?', (tid,)).fetchall()
    return f'{n} lượt; ' + ', '.join(f'{k}={v:.3g}' for k, v in ps)

rows = [{'phiên bản': 'Bài báo (Bảng 11)', 'AUROC': '72.6 ± 0.3', 'AUPRC': '23.8 ± 0.4', 'Δ AUROC so với c61fad0': ''}]
goc = ket_qua(LOGDIR)
for ten, root in [('Mã 26/05/2023 (ô 3)', KC2023), ('c61fad0 (Nhiệm vụ 1)', LOGDIR)] + \
                 [(t, f'{FIXDIR}/{s[:8]}') for s, t in COMMITS]:
    kq = ket_qua(root)
    rows.append({'phiên bản': ten,
                 'AUROC': f'{kq[0]:.1f} ± {kq[1]:.1f}' if kq else 'chưa có',
                 'AUPRC': f'{kq[2]:.1f} ± {kq[3]:.1f}' if kq else 'chưa có',
                 'Δ AUROC so với c61fad0': f'{kq[0] - goc[0]:+.1f}' if kq and goc else ''})
bang = pd.DataFrame(rows)
bang.to_csv(os.path.join(FIXDIR, 'sua_loi_lgbm.csv'), index=False)
print(f'\n{gio_vn()} 3/3 LightGBM, eICU demo, AKI, seed 1111 (AUROC / AUPRC %):')
print(bang.to_string(index=False))
print('\nSiêu tham số tốt nhất:')
for ten, root in [('c61fad0', LOGDIR)] + [(s[:8], f'{FIXDIR}/{s[:8]}') for s, t in COMMITS]:
    print(f'  {ten}: {sieu_tham_so(root)}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
00:02 1/3 Cài YAIB 26b682d7 (A: c61fad0 + sửa #200 (23/10/2026)), vài phút...
    điền 2 ca thiếu giới tính (0 nghĩa là đã điền từ trước)
00:03 1/3 Cài YAIB 6c9c19f4 (B: bản mới nhất (29/10/2026)), vài phút...
    điền 2 ca thiếu giới tính (0 nghĩa là đã điền từ trước)
00:03 2/3 26b682d7: bắt đầu, đã có 0/30 lượt tinh chỉnh
    00:13 tinh chỉnh 17/30 lượt, huấn luyện chính 0/25 fold
    00:23 tinh chỉnh 30/30 lượt, huấn luyện chính 18/25 fold
    00:24 Xong: 25/25 fold
00:24 2/3 6c9c19f4: bắt đầu, đã có 0/30 lượt tinh chỉnh
    00:34 tinh chỉnh 16/30 lượt, huấn luyện chính 0/25 fold
    00:44 tinh chỉnh 30/30 lượt, huấn luyện chính 15/25 fold
    00:46 Xong: 25/25 fold

00:46 3/3 LightGBM, eICU demo, AKI, seed 1111 (AUROC / AUPRC %):
                         phiên bản      AUROC      AUPRC Δ AUROC so với c61fad0
                 Bài báo (Bảng 11) 72.6 ± 0.3 2

In [4]:
# BỔ SUNG, ô 5. KIỂM TRA: GRU 54,5 (thấp gần ngẫu nhiên) và 63,7 (khôi phục không gian tìm kiếm 2023) có còn đúng khi đã sửa
# hai lỗi chỉ của mô hình học sâu không: #243 (trọng số lớp bị đảo khoảng một nửa số lần chạy) và #244 (chấm bằng trọng số
# epoch cuối thay vì epoch tốt nhất). Cả hai đã sửa trong commit mới nhất 6c9c19f4 (29/10/2026).
# Lượt C: 6c9c19f4, không gian tìm kiếm hiện tại   → so với c61fad0 cùng không gian: 54,5 ± 1,4
# Lượt D: 6c9c19f4, không gian tìm kiếm năm 2023   → so với c61fad0 cùng không gian: 63,7 ± 1,0
# Cách đọc: C vẫn thấp và D vẫn gần 63 thì kết luận "không gian tìm kiếm gây ra chênh lệch" giữ nguyên;
# C tăng lên gần 63 thì một phần chênh lệch là do hai lỗi trên.
# Giữ nguyên như các lần trước: dữ liệu demo eICU, seed 1111, 30 lượt tinh chỉnh × 5 fold rồi 5×5 fold, điền 2 ca thiếu giới tính.
# CẦN GPU T4 (Thời gian chạy → Thay đổi loại thời gian chạy). Khoảng 3–4 giờ cho cả hai lượt. Bị ngắt thì chạy lại ô này:
# máy chạy tiếp từ các lượt tinh chỉnh đã lưu. Log ghi vào máy Colab rồi chép sang Drive mỗi 10 phút (tránh lỗi ghi .ckpt lên Drive).
import os, re, glob, json, sqlite3, subprocess, shutil, time
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
SHA = '6c9c19f4d8e8f98addf062ea0171e97df82e745e'
REPO = f'/content/YAIB_{SHA[:8]}'                       # dùng chung thư mục với ô 4 nếu đã chạy trong phiên này
DRIVE = '/content/drive/MyDrive/YAIB_logs_gru_sualoi'
LOCAL = '/content/yaib_logs_gru_sualoi'
KHONG_GIAN_2023 = ['optimizer/hyperparameter.lr = (1e-5, 3e-4)',          # hiện tại: (1e-6, 1e-4, "log")
                   'model/hyperparameter.hidden_dim = (32, 256, "log")',  # hiện tại: (32, 512, "log")
                   'model/hyperparameter.layer_dim = (1, 3)']             # hiện tại: (1, 10)
LUOT = [('hien_tai', 'C: không gian hiện tại', [], (54.5, 1.4, 11.9, 0.6)),
        ('kg2023', 'D: không gian năm 2023', KHONG_GIAN_2023, (63.7, 1.0, 15.8, 0.5))]
NAME, TASK, MODEL = 'eicu_demo', 'AKI', 'GRU'
os.environ['MPLBACKEND'] = 'Agg'
subprocess.run(['pkill', '-f', 'icu-benchmarks train'])

gpu = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip()
assert gpu, 'CHƯA CÓ GPU: chọn Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU, rồi chạy lại ô này.'
print('GPU:', gpu)

def gio_vn():
    return time.strftime('%H:%M', time.gmtime(time.time() + 7 * 3600))

# Bước 1. Cài YAIB 6c9c19f4 (bỏ qua nếu ô 4 đã cài trong phiên này); điền 2 ca thiếu giới tính
print(f'{gio_vn()} 1/3 Cài YAIB {SHA[:8]}, vài phút...', flush=True)
os.makedirs(DRIVE, exist_ok=True)
if not os.path.isdir(f'{REPO}/.git'):
    shutil.rmtree(REPO, ignore_errors=True)
logf = os.path.join(DRIVE, 'gs1_cai.log')
with open(logf, 'w') as f:
    p = subprocess.run(f'cd /content && ([ -d {REPO} ] || git clone -q -b development https://github.com/rvandewater/YAIB.git {REPO}) && '
                       f'cd {REPO} && git checkout -q {SHA} && pip install -q uv && uv python install 3.12 && '
                       'uv sync --python 3.12 && uv run icu-benchmarks -h',
                       shell=True, executable='/bin/bash', stdout=f, stderr=subprocess.STDOUT)
if p.returncode != 0:
    print(''.join(open(logf).readlines()[-30:]))
    raise RuntimeError('Cài đặt lỗi. Gửi thông báo lỗi này cho người hướng dẫn.')
d = f'{REPO}/demo_data/aki/{NAME}'
sta = pd.read_parquet(f'{d}/sta.parquet')
n = int(sta['sex'].isna().sum())
if n:
    sta['sex'] = sta['sex'].fillna(sta['sex'].mode()[0])
    sta.to_parquet(f'{d}/sta.parquet', index=False)
    shutil.rmtree(f'{d}/cache', ignore_errors=True)
    shutil.rmtree(f'{d}/preproc', ignore_errors=True)
print(f'    điền {n} ca thiếu giới tính (0 nghĩa là đã điền từ trước)')

# Bước 2. Hai lượt GRU, mỗi lượt một thư mục log riêng
def dong_bo(src, dst, ghi_de=True):
    """Chép kết quả từ src sang dst, bỏ file mô hình .ckpt và thư mục tạm. Trả về số file chép lỗi."""
    loi = 0
    for root, dirs, files in os.walk(src):
        dirs[:] = [x for x in dirs if not x.startswith('tmp')]
        out = os.path.join(dst, os.path.relpath(root, src))
        os.makedirs(out, exist_ok=True)
        for fn in files:
            if fn.endswith('.ckpt'):
                continue
            s, t = os.path.join(root, fn), os.path.join(out, fn)
            try:
                if os.path.exists(t) and (not ghi_de or (not fn.endswith('.db') and os.path.getsize(t) == os.path.getsize(s)
                                                         and os.path.getmtime(t) >= os.path.getmtime(s) - 2)):
                    continue
                if fn.endswith('.db'):  # file SQLite (có thể đang được ghi): sao lưu qua API của SQLite
                    a, b = sqlite3.connect(s), sqlite3.connect('/tmp/_ban_sao.db')
                    a.backup(b)
                    a.close(); b.close()
                    shutil.copyfile('/tmp/_ban_sao.db', t)
                else:
                    shutil.copy2(s, t)
            except Exception:
                loi += 1
    return loi

def so_fold(run):
    return len(glob.glob(os.path.join(run, 'repetition_*', 'fold_*', 'test_metrics.json'))) if run else 0

def final_run(root):
    runs = [os.path.dirname(p) for p in glob.glob(os.path.join(root, NAME, TASK, MODEL, '*', 'accumulated_test_metrics.json'))]
    runs = [r for r in runs if so_fold(r) == 25]
    return max(runs, key=os.path.getmtime) if runs else None

def best_db(base):
    best, best_n = None, 0
    for db in glob.glob(os.path.join(base, '*', 'hyperparameter_tuning_logs.db')):
        try:
            n = sqlite3.connect(db).execute("select count(*) from trials where state='COMPLETE'").fetchone()[0]
        except Exception:
            n = 0
        if best is None or n > best_n:
            best, best_n = db, n
    return best, best_n

def khoang(db):
    """Khoảng tốc độ học và số tầng lớn nhất đã thử, để kiểm tra khoảng tìm kiếm có được áp dụng đúng."""
    try:
        con = sqlite3.connect(db)
        lo, hi = con.execute("select min(param_value), max(param_value) from trial_params where param_name like '%lr'").fetchone()
        tang = con.execute("select max(param_value) from trial_params where param_name like '%layer_dim'").fetchone()[0]
        return f'(đã thử lr {lo:.1e}–{hi:.1e}, tối đa {int(tang)} tầng)'
    except Exception:
        return ''

def chay(khoa, bindings):
    local, drv = f'{LOCAL}/{khoa}', f'{DRIVE}/{khoa}'
    base, base_drive = os.path.join(local, NAME, TASK, MODEL), os.path.join(drv, NAME, TASK, MODEL)
    dong_bo(base_drive, base, ghi_de=False)        # lấy các lượt đã có trên Drive về máy Colab
    db, n = best_db(base)
    cmd = ['uv', 'run', 'icu-benchmarks', 'train', '-d', f'demo_data/aki/{NAME}', '-n', NAME,
           '-t', 'BinaryClassification', '-tn', TASK, '-m', MODEL, '--seed', '1111',
           '-l', local, '--tune', '-gc', '-lc']
    if db:
        cmd += ['--hp-checkpoint', os.path.relpath(db, base)]
    if bindings:
        cmd += ['-hp', *bindings]
    print(f'{gio_vn()} 2/3 {khoa}: bắt đầu, đã có {n}/30 lượt tinh chỉnh', flush=True)
    os.makedirs(local, exist_ok=True)
    logf = os.path.join(local, f'gs2_{khoa}_{time.strftime("%m%d-%H%M")}.log')
    with open(logf, 'w') as f:
        p = subprocess.Popen(cmd, cwd=REPO, stdout=f, stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    p.wait(timeout=600)
                    break
                except subprocess.TimeoutExpired:
                    dong_bo(base, base_drive)          # chép sang Drive mỗi 10 phút
                    runs = sorted(glob.glob(os.path.join(base, '20*')))
                    db_now = best_db(base)
                    print(f'    {gio_vn()} tinh chỉnh {db_now[1]}/30 lượt, '
                          f'huấn luyện chính {so_fold(runs[-1]) if runs else 0}/25 fold {khoang(db_now[0])}', flush=True)
        except KeyboardInterrupt:
            subprocess.run(['pkill', '-f', 'icu-benchmarks train'])
            dong_bo(base, base_drive)
            raise
    for _ in range(5):                                 # chép lần cuối, thử lại nếu Drive báo lỗi
        if dong_bo(base, base_drive) == 0:
            break
        time.sleep(30)
    try:
        shutil.copy2(logf, drv)
    except Exception:
        pass
    if p.returncode != 0:
        print(f'    LỖI (mã {p.returncode}). 30 dòng cuối của log:')
        print(''.join(open(logf).readlines()[-30:]))
        return False
    print(f'    {gio_vn()} Xong: {so_fold(final_run(drv))}/25 fold đã lưu trên Drive', flush=True)
    return True

for khoa, ten, bindings, _ in LUOT:
    for lan in range(5):   # thử lại nếu Drive lỗi tạm thời
        if final_run(f'{DRIVE}/{khoa}'):
            print(f'{gio_vn()} 2/3 {khoa}: đã xong 25/25 fold, bỏ qua')
            break
        if chay(khoa, bindings):
            break
        print(f'    {gio_vn()} chờ 1 phút rồi chạy tiếp...', flush=True)
        time.sleep(60)

# Bước 3. Bảng so sánh (± là sai số chuẩn qua 25 fold) và cấu hình được chọn
rows = [{'lượt': 'Bài báo (Bảng 11)', 'AUROC': '63.0 ± 0.8', 'AUPRC': '17.3 ± 0.6', 'cùng không gian, mã c61fad0': ''}]
for khoa, ten, bindings, cu in LUOT:
    run = final_run(f'{DRIVE}/{khoa}')
    m = json.load(open(os.path.join(run, 'accumulated_test_metrics.json'))) if run else None
    rows.append({'lượt': ten + ' (6c9c19f4)',
                 'AUROC': f"{100*m['avg']['AUC']:.1f} ± {100*m['std']['AUC']:.1f}" if m else 'chưa có',
                 'AUPRC': f"{100*m['avg']['PR']:.1f} ± {100*m['std']['PR']:.1f}" if m else 'chưa có',
                 'cùng không gian, mã c61fad0': f'{cu[0]} ± {cu[1]} / {cu[2]} ± {cu[3]}'})
bang = pd.DataFrame(rows)
bang.to_csv(os.path.join(DRIVE, 'gru_sua_loi.csv'), index=False)
print(f'\n{gio_vn()} 3/3 GRU, eICU demo, AKI, seed 1111 (AUROC / AUPRC %):')
print(bang.to_string(index=False))
for khoa, ten, bindings, _ in LUOT:
    run = final_run(f'{DRIVE}/{khoa}')
    if not run:
        continue
    cfg = open(os.path.join(run, 'repetition_0', 'fold_0', 'train_config.gin')).read().splitlines()
    print(f'\n{ten}, cấu hình được chọn {khoang(best_db(os.path.join(DRIVE, khoa, NAME, TASK, MODEL))[0])}:')
    print('\n'.join('    ' + l for l in cfg if re.search(r'hidden_dim|layer_dim|\blr\b', l)))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
04:49 1/3 Cài YAIB 6c9c19f4, vài phút...
    điền 0 ca thiếu giới tính (0 nghĩa là đã điền từ trước)
04:49 2/3 hien_tai: đã xong 25/25 fold, bỏ qua
04:49 2/3 kg2023: đã xong 25/25 fold, bỏ qua

04:49 3/3 GRU, eICU demo, AKI, seed 1111 (AUROC / AUPRC %):
                             lượt      AUROC      AUPRC cùng không gian, mã c61fad0
                Bài báo (Bảng 11) 63.0 ± 0.8 17.3 ± 0.6                            
C: không gian hiện tại (6c9c19f4) 70.4 ± 0.6 21.9 ± 0.7     54.5 ± 1.4 / 11.9 ± 0.6
D: không gian năm 2023 (6c9c19f4) 70.3 ± 0.6 21.9 ± 0.6     63.7 ± 1.0 / 15.8 ± 0.5

C: không gian hiện tại, cấu hình được chọn (đã thử lr 1.5e-06–1.0e-04, tối đa 10 tầng):
    Adam.lr = 0.0001
    DLPredictionWrapper.lr = 0.002
    GRUNet.hidden_dim = 331
    GRUNet.layer_dim = 1
    model/hyperparameter.hidden_dim = (32, 512, 'log')
    model/hype